# MAESTRO TRADING SYSTEM v3.0
## Time-Cycle | Screener Multi-Aset | Chart Interaktif | Berita & Sentimen | Plan Trading 1 Bulan

---

> **Filosofi Inti:** Harga bergerak dalam siklus waktu yang dapat diukur secara matematis.

---

### PANDUAN PENGGUNAAN

```
+--------------------+------------------------------------------+
| Mode               | Cell yang Dijalankan                     |
+--------------------+------------------------------------------+
| Analisis 1 Aset    | Cell 1 -> 7  lalu  Cell 8A               |
| Screener Saham IDX | Cell 1 -> 7  lalu  Cell 8B               |
| Screener Crypto    | Cell 1 -> 7  lalu  Cell 8C               |
| Screener Forex     | Cell 1 -> 7  lalu  Cell 8D               |
+--------------------+------------------------------------------+
```

> **WAJIB: Selalu jalankan Cell 1 s/d Cell 7 dulu sebelum Cell 8!**

---

In [3]:
# ======================================================================
# CELL 1: INSTALASI & IMPORT LIBRARY
# Jalankan cell ini PERTAMA KALI untuk menyiapkan seluruh mesin.
# ======================================================================

import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

for lib in ["yfinance", "pandas", "plotly", "scipy", "numpy",
            "requests", "beautifulsoup4", "lxml", "html5lib"]:
    install(lib)

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import requests
import json
import re
import time
import random
from datetime import datetime, timedelta
from urllib.parse import quote

import yfinance as yf
from scipy.signal import find_peaks
from bs4 import BeautifulSoup

import plotly.graph_objects as go
from plotly.subplots import make_subplots
from IPython.display import display, HTML

print("=" * 65)
print("  MAESTRO ENGINE v3.0  -  SEMUA SENJATA SIAP")
print("=" * 65)
print(f"  [OK]  numpy      {np.__version__}")
print(f"  [OK]  pandas     {pd.__version__}")
print(f"  [OK]  yfinance   {yf.__version__}")
print(f"  [OK]  scipy      (find_peaks aktif)")
print(f"  [OK]  requests   (web scraping aktif)")
print(f"  [OK]  plotly     (chart interaktif aktif)")
print("=" * 65)
print("  Siap untuk analisis siklus waktu & multi-aset screening!")
print("=" * 65)

  MAESTRO ENGINE v3.0  -  SEMUA SENJATA SIAP
  [OK]  numpy      2.4.2
  [OK]  pandas     3.0.0
  [OK]  yfinance   1.3.0
  [OK]  scipy      (find_peaks aktif)
  [OK]  requests   (web scraping aktif)
  [OK]  plotly     (chart interaktif aktif)
  Siap untuk analisis siklus waktu & multi-aset screening!


---
## Cell 2 - Engine Penarik Data & Auto-Konversi IDR

**Perbaikan v3:**
- Fix MultiIndex columns untuk semua aset (crypto, forex, saham US)
- Tambah retry otomatis jika data kosong
- Support semua jenis ticker tanpa error


In [ ]:
# ======================================================================
# CELL 2: ENGINE PENARIK DATA & AUTO-KONVERSI IDR
# FIX v3: MultiIndex handling + retry logic + support semua aset
# ======================================================================

def flatten_columns(df):
    """Ratakan MultiIndex columns jadi flat - fix utama untuk crypto & forex."""
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    return df


def ambil_kurs_idr():
    """Ambil kurs USD/IDR terkini dari yfinance."""
    for ticker_kurs in ["IDR=X", "USDIDR=X"]:
        try:
            kurs_df = yf.download(ticker_kurs, period="5d", interval="1d", progress=False)
            kurs_df = flatten_columns(kurs_df)
            if not kurs_df.empty and "Close" in kurs_df.columns:
                kurs = float(kurs_df["Close"].dropna().iloc[-1])
                if kurs > 1000:  # validasi: kurs IDR harus > 1000
                    return kurs
        except Exception:
            continue
    return 16200.0  # fallback 2025


def tarik_data_maestro(ticker: str, period: str = "1y") -> dict:
    """
    Tarik data OHLCV lengkap dari yfinance untuk SEMUA jenis aset.
    Fix v3: Tidak lagi error untuk crypto, forex, saham US.
    """
    ticker = ticker.strip().upper()
    is_jk = ticker.upper().endswith(".JK")

    # --- Deteksi jenis aset ---
    is_crypto = any(ticker.endswith(x) for x in ["-USD", "-USDT", "-BTC"])
    is_forex  = ticker.endswith("=X")

    # --- Tarik data dengan retry ---
    raw = None
    for attempt in range(3):
        try:
            raw = yf.download(ticker, period=period, interval="1d",
                              progress=False, auto_adjust=True)
            if not raw.empty:
                break
            # Jika kosong, coba period lebih pendek
            if attempt == 1:
                raw = yf.download(ticker, period="6mo", interval="1d",
                                  progress=False, auto_adjust=True)
        except Exception:
            time.sleep(0.5)
            continue

    if raw is None or raw.empty:
        raise ValueError(f"Tidak ada data untuk ticker: {ticker}. "
                         f"Pastikan kode benar (contoh: BTC-USD, EURUSD=X, BBCA.JK, AAPL)")

    # --- Flatten MultiIndex (fix utama!) ---
    raw = flatten_columns(raw)

    # --- Pastikan kolom standar ada ---
    kolom_wajib = ["Open", "High", "Low", "Close"]
    for k in kolom_wajib:
        if k not in raw.columns:
            raise ValueError(f"Kolom '{k}' tidak ditemukan. Data tidak valid untuk {ticker}.")

    # --- Reset index & normalisasi Date ---
    df = raw.reset_index().copy()
    if "index" in df.columns:
        df.rename(columns={"index": "Date"}, inplace=True)
    if "Datetime" in df.columns:
        df.rename(columns={"Datetime": "Date"}, inplace=True)
    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()

    # --- Konversi IDR ---
    kurs = 1.0
    if is_jk:
        for col in ["Open", "High", "Low", "Close"]:
            df[f"{col}_IDR"] = pd.to_numeric(df[col], errors="coerce")
        kurs_label = "IDR (langsung)"
    else:
        kurs = ambil_kurs_idr()
        for col in ["Open", "High", "Low", "Close"]:
            df[f"{col}_IDR"] = pd.to_numeric(df[col], errors="coerce") * kurs
        kurs_label = f"IDR (kurs: Rp {kurs:,.0f})"

    # --- Volume (aman untuk forex yang tidak punya volume) ---
    if "Volume" in df.columns:
        df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce").fillna(0)
    else:
        df["Volume"] = 0.0

    df = df.dropna(subset=["Close_IDR"]).sort_values("Date").reset_index(drop=True)

    if len(df) < 5:
        raise ValueError(f"Data untuk {ticker} terlalu sedikit ({len(df)} baris). "
                         "Coba period lebih panjang atau ticker lain.")

    harga_terakhir = float(df["Close_IDR"].iloc[-1])
    harga_kemarin  = float(df["Close_IDR"].iloc[-2]) if len(df) > 1 else harga_terakhir
    pct_change     = (harga_terakhir - harga_kemarin) / max(harga_kemarin, 1) * 100

    # --- Label jenis aset ---
    if is_crypto:   jenis = "[CRYPTO]"
    elif is_forex:  jenis = "[FOREX] "
    elif is_jk:     jenis = "[IDX]   "
    else:           jenis = "[SAHAM] "

    tanda = "+" if pct_change >= 0 else ""
    print(f"  {jenis} [{ticker:<14}]  Rp {harga_terakhir:>16,.2f}  "
          f"({tanda}{pct_change:.2f}%)  -  {kurs_label}")

    return {
        "df":                  df,
        "kurs":                kurs,
        "is_jk":               is_jk,
        "is_crypto":           is_crypto,
        "is_forex":            is_forex,
        "harga_terakhir_idr":  harga_terakhir,
        "pct_change":          pct_change,
        "ticker":              ticker,
        "kurs_label":          kurs_label,
    }


import time, random

def tarik_data_batch(daftar_ticker, period="1y", batch_size=50):
    hasil = {}
    # Bagi jadi batch kecil
    for i in range(0, len(daftar_ticker), batch_size):
        batch = daftar_ticker[i:i+batch_size]
        for t in batch:
            try:
                hasil[t] = tarik_data_maestro(t, period=period)
            except Exception as e:
                print(f"  [SKIP] {t}: {str(e)[:50]}")
            # Jeda acak antar request — kunci anti-block
            time.sleep(random.uniform(0.2, 0.6))
        # Jeda lebih lama antar batch
        if i + batch_size < len(daftar_ticker):
            print(f"  [JEDA] Batch selesai, tunggu 3 detik...")
            time.sleep(3)
    return hasil


print("[OK] Cell 2 - Engine Penarik Data (Fix Multi-Aset) berhasil dimuat.")

[OK] Cell 2 - Engine Penarik Data (Fix Multi-Aset) berhasil dimuat.


---
## Cell 3 - Engine Matematika Siklus Waktu

In [5]:
# ======================================================================
# CELL 3: ENGINE MATEMATIKA SIKLUS WAKTU
# Prinsip: Price-Time Squaring & Cartesian Coordinate Analysis
# ======================================================================

def deteksi_siklus_waktu(df: pd.DataFrame, prominence_pct: float = 0.02) -> dict:
    """
    Deteksi puncak & lembah secara kuantitatif menggunakan scipy.
    Proyeksikan reversal berikutnya + plan trading 1 bulan ke depan.
    """
    close = df["Close_IDR"].values
    dates = pd.to_datetime(df["Date"].values)

    price_range = close.max() - close.min()
    if price_range == 0:
        price_range = close.mean() * 0.1
    prominence = price_range * prominence_pct

    peaks,   _ = find_peaks(close,  prominence=prominence, distance=5)
    valleys, _ = find_peaks(-close, prominence=prominence, distance=5)

    def hitung_siklus(indices, dates_arr):
        if len(indices) < 2:
            return None, []
        gaps = [(dates_arr[indices[i+1]] - dates_arr[indices[i]]).days
                for i in range(len(indices)-1)]
        return int(np.mean(gaps)), gaps

    avg_peak_cycle,   peak_gaps   = hitung_siklus(peaks,   dates)
    avg_valley_cycle, valley_gaps = hitung_siklus(valleys, dates)

    today = dates[-1]
    next_peak_date   = None
    next_valley_date = None

    if len(peaks) > 0 and avg_peak_cycle:
        last_peak_date    = dates[peaks[-1]]
        days_since_peak   = (today - last_peak_date).days
        days_to_next_peak = avg_peak_cycle - (days_since_peak % avg_peak_cycle)
        next_peak_date    = today + timedelta(days=int(days_to_next_peak))

    if len(valleys) > 0 and avg_valley_cycle:
        last_valley_date    = dates[valleys[-1]]
        days_since_valley   = (today - last_valley_date).days
        days_to_next_valley = avg_valley_cycle - (days_since_valley % avg_valley_cycle)
        next_valley_date    = today + timedelta(days=int(days_to_next_valley))

    # --- Buat jadwal siklus 30 hari ke depan ---
    jadwal_30hari = buat_jadwal_30hari(today, avg_peak_cycle, avg_valley_cycle,
                                       next_peak_date, next_valley_date)

    df_ann = df.copy()
    df_ann["Point_Type"] = ""
    for idx in peaks:
        if idx < len(df_ann):
            df_ann.at[idx, "Point_Type"] = "PEAK"
    for idx in valleys:
        if idx < len(df_ann):
            df_ann.at[idx, "Point_Type"] = "VALLEY"

    return {
        "peaks_idx":        peaks,
        "valleys_idx":      valleys,
        "avg_peak_cycle":   avg_peak_cycle,
        "avg_valley_cycle": avg_valley_cycle,
        "peak_gaps":        peak_gaps,
        "valley_gaps":      valley_gaps,
        "next_peak_date":   next_peak_date,
        "next_valley_date": next_valley_date,
        "jadwal_30hari":    jadwal_30hari,
        "df_annotated":     df_ann,
    }


def buat_jadwal_30hari(today, avg_peak, avg_valley, next_peak, next_valley):
    """
    Buat daftar event trading untuk 30 hari ke depan berdasarkan siklus data.
    Return list of dict: {tanggal, event, aksi, prioritas}
    """
    jadwal = []

    # Generate semua puncak & lembah dalam 30 hari ke depan
    if next_peak and avg_peak:
        t = pd.Timestamp(next_peak)
        end = pd.Timestamp(today) + pd.Timedelta(days=30)
        while t <= end:
            jadwal.append({
                "tanggal":   t,
                "event":     "PUNCAK SIKLUS",
                "aksi":      "JUAL / TAKE PROFIT",
                "prioritas": "TINGGI",
                "type":      "peak",
            })
            t += pd.Timedelta(days=avg_peak)

    if next_valley and avg_valley:
        t = pd.Timestamp(next_valley)
        end = pd.Timestamp(today) + pd.Timedelta(days=30)
        while t <= end:
            jadwal.append({
                "tanggal":   t,
                "event":     "LEMBAH SIKLUS",
                "aksi":      "BELI / AKUMULASI",
                "prioritas": "TINGGI",
                "type":      "valley",
            })
            t += pd.Timedelta(days=avg_valley)

    jadwal.sort(key=lambda x: x["tanggal"])
    return jadwal


def print_cycle_summary(ticker: str, cycle: dict):
    print("\n" + "=" * 65)
    print(f"  ANALISIS SIKLUS WAKTU -- {ticker}")
    print("=" * 65)
    pk  = cycle["avg_peak_cycle"]
    vl  = cycle["avg_valley_cycle"]
    npk = cycle["next_peak_date"]
    nvl = cycle["next_valley_date"]

    if pk:
        print(f"  [UP]   Rata-rata Siklus PUNCAK  : setiap {pk} hari")
        print(f"  [CAL]  Prediksi Puncak Berikutnya : {npk.strftime('%d %b %Y') if npk else 'N/A'}")
    else:
        print("  [!]   Puncak: data historis terlalu sedikit untuk proyeksi.")

    if vl:
        print(f"  [DN]   Rata-rata Siklus LEMBAH  : setiap {vl} hari")
        print(f"  [CAL]  Prediksi Lembah Berikutnya : {nvl.strftime('%d %b %Y') if nvl else 'N/A'}")
    else:
        print("  [!]   Lembah: data historis terlalu sedikit untuk proyeksi.")

    print("=" * 65)
    print(f"  Jumlah Puncak Terdeteksi : {len(cycle['peaks_idx'])}")
    print(f"  Jumlah Lembah Terdeteksi : {len(cycle['valleys_idx'])}")
    print("=" * 65 + "\n")


print("[OK] Cell 3 - Engine Siklus Waktu berhasil dimuat.")

[OK] Cell 3 - Engine Siklus Waktu berhasil dimuat.


---
## Cell 4 - Engine Screener Fundamental & Filter Budget

In [6]:
# ======================================================================
# CELL 4: ENGINE SCREENER FUNDAMENTAL & FILTER BUDGET
# ======================================================================

def ambil_fundamental(ticker: str) -> dict:
    """Tarik data fundamental dari yfinance.info."""
    try:
        info = yf.Ticker(ticker).info
        return {
            "PER":           info.get("trailingPE",              None),
            "PBV":           info.get("priceToBook",             None),
            "ROE":           info.get("returnOnEquity",          None),
            "Profit_Margin": info.get("profitMargins",           None),
            "Debt_Equity":   info.get("debtToEquity",            None),
            "Market_Cap":    info.get("marketCap",               None),
            "Sector":        info.get("sector", info.get("category", "--")),
            "Industry":      info.get("industry", "--"),
            "Nama":          info.get("longName", info.get("shortName", ticker)),
            "EPS":           info.get("trailingEps",             None),
            "Beta":          info.get("beta",                    None),
            "DivYield":      info.get("dividendYield",           None),
            "52W_High":      info.get("fiftyTwoWeekHigh",        None),
            "52W_Low":       info.get("fiftyTwoWeekLow",         None),
            "Deskripsi":     (info.get("longBusinessSummary", "Tidak tersedia.") or "")[:400],
        }
    except Exception:
        return {k: None for k in ["PER", "PBV", "ROE", "Profit_Margin",
                                   "Debt_Equity", "Market_Cap", "Sector",
                                   "Industry", "Nama", "EPS", "Beta",
                                   "DivYield", "52W_High", "52W_Low", "Deskripsi"]}


def hitung_skor_fundamental(f: dict) -> float:
    """
    Scoring sistem 0-100 berbasis kesehatan finansial.
    Komponen: PER (25) | PBV (20) | ROE (25) | Margin (15) | Debt/EQ (15)
    """
    skor = 0.0

    per = f.get("PER")
    if per and per > 0:
        if   per < 10: skor += 25
        elif per < 15: skor += 22
        elif per < 20: skor += 17
        elif per < 25: skor += 12
        elif per < 40: skor += 5
        else:           skor += 0
    else:
        skor += 10

    pbv = f.get("PBV")
    if pbv and pbv > 0:
        if   pbv < 1:  skor += 20
        elif pbv < 2:  skor += 17
        elif pbv < 3:  skor += 12
        elif pbv < 5:  skor += 6
        else:           skor += 0
    else:
        skor += 8

    roe = f.get("ROE")
    if roe is not None:
        rp = roe * 100
        if   rp >= 25: skor += 25
        elif rp >= 15: skor += 20
        elif rp >= 10: skor += 14
        elif rp >= 5:  skor += 7
        else:          skor += 0
    else:
        skor += 8

    margin = f.get("Profit_Margin")
    if margin is not None:
        mp = margin * 100
        if   mp >= 25: skor += 15
        elif mp >= 15: skor += 12
        elif mp >= 8:  skor += 8
        elif mp >= 3:  skor += 4
        else:          skor += 0
    else:
        skor += 5

    de = f.get("Debt_Equity")
    if de is not None:
        if   de < 30:  skor += 15
        elif de < 60:  skor += 12
        elif de < 100: skor += 7
        elif de < 150: skor += 3
        else:          skor += 0
    else:
        skor += 5

    return round(min(skor, 100), 1)


def screener_budget(data_batch: dict, budget: float) -> pd.DataFrame:
    print("\n" + "=" * 70)
    print(f"  SCREENER BUDGET -- Modal Tersedia: Rp {budget:,.0f}")
    print("=" * 70)

    rows = []
    for ticker, result in data_batch.items():
        harga = result["harga_terakhir_idr"]
        is_jk = result["is_jk"]

        if is_jk:
            harga_1lot   = harga * 100
            satuan_label = "1 Lot (100 lbr)"
        else:
            harga_1lot   = harga
            satuan_label = "1 Unit"

        lolos_budget = harga_1lot <= budget
        status       = "LOLOS" if lolos_budget else "MAHAL"
        simbol       = "[v]" if lolos_budget else "[x]"

        print(f"  {simbol} [{ticker:<14}] Rp {harga:>16,.2f} | "
              f"{satuan_label}: Rp {harga_1lot:>14,.2f} -> {status}")

        fund = ambil_fundamental(ticker)
        skor = hitung_skor_fundamental(fund) if lolos_budget else 0.0

        rows.append({
            "Ticker":         ticker,
            "Nama":           (fund.get("Nama") or ticker)[:30],
            "Sektor":         fund.get("Sector") or "--",
            "Harga (IDR)":    round(harga, 2),
            "Min. Beli (IDR)": round(harga_1lot, 2),
            "Budget OK?":     "[v]" if lolos_budget else "[x]",
            "PER":            round(fund["PER"], 1) if fund.get("PER") else "--",
            "PBV":            round(fund["PBV"], 2) if fund.get("PBV") else "--",
            "ROE (%)": f"{fund['ROE']*100:.1f}%" if fund.get("ROE") else "--",
            "Skor (0-100)":   skor,
        })

    df_hasil = pd.DataFrame(rows).sort_values("Skor (0-100)",
                ascending=False).reset_index(drop=True)
    df_hasil.index += 1
    print("\n  Screener selesai. Tabel diurutkan dari skor tertinggi.")
    print("=" * 70 + "\n")
    return df_hasil


def cetak_analisis_keuangan_lengkap(ticker, fund, is_jk=False):
    print(f"\n{'='*65}")
    print(f"  ANALISIS KEUANGAN LENGKAP -- {ticker}")
    print(f"{'='*65}")

    nama      = fund.get("Nama") or ticker
    sektor    = fund.get("Sector") or "--"
    industri  = fund.get("Industry") or "--"
    deskripsi = fund.get("Deskripsi") or "Tidak tersedia."

    print(f"  [CO] {nama}")
    print(f"  [SE] Sektor   : {sektor}")
    print(f"  [IN] Industri : {industri}")
    print(f"  [NB] Profil   : {deskripsi[:200]}...")
    print()

    print("  -- VALUASI " + "-" * 52)
    per = fund.get("PER")
    pbv = fund.get("PBV")

    if per:
        label_per = "[Murah]" if per < 15 else ("[Wajar]" if per < 25 else "[Mahal]")
        print(f"  PER (P/E Ratio)    : {per:.1f}x  {label_per}")
    else:
        print("  PER (P/E Ratio)    : --")

    if pbv:
        label_pbv = "[Undervalued]" if pbv < 1 else ("[Wajar]" if pbv < 3 else "[Premium]")
        print(f"  PBV (Price/Book)   : {pbv:.2f}x  {label_pbv}")
    else:
        print("  PBV (Price/Book)   : --")

    print()
    print("  -- PROFITABILITAS " + "-" * 45)
    roe    = fund.get("ROE")
    margin = fund.get("Profit_Margin")
    eps    = fund.get("EPS")

    if roe:
        rp = roe * 100
        rt = "[Sangat Baik]" if rp >= 20 else ("[Cukup]" if rp >= 10 else "[Lemah]")
        print(f"  ROE                : {rp:.1f}%  {rt}")
    else:
        print("  ROE                : --")

    if margin:
        mp = margin * 100
        mt = "[Efisien]" if mp >= 15 else ("[Cukup]" if mp >= 5 else "[Tipis]")
        print(f"  Net Profit Margin  : {mp:.1f}%  {mt}")
    else:
        print("  Net Profit Margin  : --")

    if eps:
        print(f"  EPS (Laba/Saham)   : {eps:.2f}")

    print()
    print("  -- KESEHATAN KEUANGAN " + "-" * 41)
    de   = fund.get("Debt_Equity")
    beta = fund.get("Beta")
    div  = fund.get("DivYield")

    if de is not None:
        dt = "[Rendah]" if de < 60 else ("[Moderat]" if de < 120 else "[Tinggi]")
        print(f"  Debt/Equity        : {de:.0f}%  {dt}")
    else:
        print("  Debt/Equity        : --")

    if beta:
        vt = "sangat rendah" if beta < 0.5 else ("rendah" if beta < 1 else
             "normal" if beta < 1.5 else "tinggi")
        print(f"  Beta (Volatilitas) : {beta:.2f}  ({vt} vs market)")

    if div:
        print(f"  Dividend Yield     : {div*100:.2f}%")

    w52h = fund.get("52W_High")
    w52l = fund.get("52W_Low")
    if w52h and w52l:
        print()
        print("  -- 52 WEEK RANGE " + "-" * 46)
        print(f"  52W High           : {w52h:,.4f}")
        print(f"  52W Low            : {w52l:,.4f}")

    skor = hitung_skor_fundamental(fund)
    print()
    print(f"  -- SKOR FUNDAMENTAL MAESTRO: {skor}/100 " + "-" * 27)
    if   skor >= 75: print("  [EXCELLENT] Fundamental sangat kuat")
    elif skor >= 55: print("  [GOOD]      Fundamental baik, layak dipertimbangkan")
    elif skor >= 35: print("  [FAIR]      Cukup, perlu monitoring")
    else:            print("  [WEAK]      Fundamental lemah, risiko tinggi")
    print(f"{'='*65}\n")


print("[OK] Cell 4 - Engine Screener Fundamental berhasil dimuat.")

[OK] Cell 4 - Engine Screener Fundamental berhasil dimuat.


---
## Cell 5 - Engine Sinyal Trading & Bandarmologi

In [7]:
# ======================================================================
# CELL 5: ENGINE PEMICU SINYAL & BANDARMOLOGI
# ======================================================================

def hitung_atr(df: pd.DataFrame, window: int = 14) -> pd.Series:
    h = df["High_IDR"]
    l = df["Low_IDR"]
    c = df["Close_IDR"]
    c_prev = c.shift(1)
    tr = pd.concat([h - l, (h - c_prev).abs(), (l - c_prev).abs()], axis=1).max(axis=1)
    return tr.rolling(window).mean()


def deteksi_sinyal_semua(df: pd.DataFrame, cycle: dict) -> pd.DataFrame:
    df = df.copy()
    n  = len(df)

    atr         = hitung_atr(df)
    atr_mean    = atr.rolling(20).mean()
    daily_range = df["High_IDR"] - df["Low_IDR"]

    vol_mean  = df["Volume"].rolling(20).mean()
    vol_spike = df["Volume"] > (vol_mean * 1.5)

    # 1. SINYAL SCALPING
    df["Sinyal_Scalping"] = ((daily_range > atr_mean * 1.5) & vol_spike).apply(
        lambda x: ">> SCALP" if x else "--")

    # 2. SINYAL SWING (berbasis siklus)
    swing_col = ["--"] * n
    for c_type, p_date in [("BELI", cycle.get("next_valley_date")),
                           ("JUAL", cycle.get("next_peak_date"))]:
        if p_date:
            p_ts = pd.Timestamp(p_date)
            for i, r_date in enumerate(df["Date"]):
                if abs((pd.Timestamp(r_date) - p_ts).days) <= 3:
                    swing_col[i] = f">> SWING {c_type}"
    df["Sinyal_Swing"] = swing_col

    # 3. SINYAL BPJS (Beli Pagi Jual Sore)
    df["Sinyal_BPJS"] = (((df["Open_IDR"] - df["Low_IDR"]) / df["Low_IDR"].replace(0, np.nan)) <= 0.003).apply(
        lambda x: ">> BPJS" if x else "--")

    # 4. SINYAL BSJP (Beli Sore Jual Pagi)
    df["Sinyal_BSJP"] = (((df["Close_IDR"] / df["High_IDR"].replace(0, np.nan)) >= 0.97) & vol_spike).apply(
        lambda x: ">> BSJP" if x else "--")

    # 5. BANDARMOLOGI
    is_up   = df["Close_IDR"] > df["Open_IDR"]
    is_down = df["Close_IDR"] < df["Open_IDR"]
    bandar_col = []
    for i in range(n):
        if vol_spike.iloc[i] and is_up.iloc[i]:
            bandar_col.append("[+] AKUMULASI (Bandar Masuk)")
        elif vol_spike.iloc[i] and is_down.iloc[i]:
            bandar_col.append("[-] DISTRIBUSI (Bandar Buang)")
        else:
            bandar_col.append("[ ] NETRAL")
    df["Sinyal_Bandarmologi"] = bandar_col

    return df


def ringkasan_sinyal(df: pd.DataFrame, ticker: str):
    cols    = ["Date", "Close_IDR", "Sinyal_Bandarmologi",
               "Sinyal_Scalping", "Sinyal_Swing", "Sinyal_BPJS", "Sinyal_BSJP"]
    terbaru = df[cols].tail(5).copy()
    terbaru["Close_IDR"] = terbaru["Close_IDR"].apply(lambda x: f"Rp {x:,.0f}")
    terbaru["Date"]      = terbaru["Date"].dt.strftime("%d %b %Y")
    print(f"\n{'='*85}")
    print(f"  RINGKASAN SINYAL & BANDARMOLOGI -- {ticker}")
    print(f"{'='*85}")
    display(terbaru.reset_index(drop=True))


print("[OK] Cell 5 - Engine Sinyal & Bandarmologi berhasil dimuat.")

[OK] Cell 5 - Engine Sinyal & Bandarmologi berhasil dimuat.


---
## Cell 6 - Visualisasi Chart TradingView-Style + Plan Trading

**Fix v3:**
- Output print rapi tanpa emoji berlebihan
- Plan trading 1 bulan ke depan berbasis data siklus asli (bukan hardcode)
- Tabel jadwal trading terstruktur

In [8]:
# ======================================================================
# CELL 6: VISUALISASI CHART INTERAKTIF + PLAN TRADING 1 BULAN
# ======================================================================

def tambah_vline_manual(fig, x_date, color, dash, label):
    x_str = pd.Timestamp(x_date).strftime("%Y-%m-%d")
    fig.add_shape(
        type="line", x0=x_str, x1=x_str, y0=0, y1=1,
        yref="paper", xref="x",
        line=dict(color=color, dash=dash, width=1.5),
    )
    fig.add_annotation(
        x=x_str, y=1.02, yref="paper", xref="x",
        text=label, showarrow=False,
        font=dict(color=color, size=11), xanchor="center",
    )


def hitung_sr_kuat(df, lookback=20, n_level=3):
    low  = df["Low_IDR"].values
    high = df["High_IDR"].values
    sup_pts, res_pts = [], []
    for i in range(lookback, len(df)):
        sup_pts.append(float(low[i-lookback:i].min()))
        res_pts.append(float(high[i-lookback:i].max()))

    def cluster(pts, n):
        if not pts: return []
        pts = sorted(set(round(p, -2) for p in pts))
        clusters, used = [], set()
        for p in pts:
            if p in used: continue
            group = [x for x in pts if abs(x-p)/max(p, 1) < 0.02]
            clusters.append(float(sum(group)/len(group)))
            for x in group: used.add(x)
        return sorted(clusters)[-n:]

    return cluster(sup_pts, n_level), cluster(res_pts, n_level)


def buat_chart_maestro(result, cycle, df_signals):
    ticker = result["ticker"]
    df     = df_signals.copy()
    dates  = df["Date"]
    close  = df["Close_IDR"]

    lookback = 20
    df["Support"]    = df["Low_IDR"].rolling(lookback).min().shift(1)
    df["Resistance"] = df["High_IDR"].rolling(lookback).max().shift(1)

    sup_levels, res_levels = hitung_sr_kuat(df)

    fig = make_subplots(
        rows=2, cols=1, row_heights=[0.75, 0.25],
        shared_xaxes=True,
        subplot_titles=(f"{ticker} -- Price Chart (IDR)", "Volume"),
        vertical_spacing=0.05,
    )

    fig.add_trace(go.Scatter(
        x=list(dates)+list(dates[::-1]),
        y=list(df["Support"])+list(df["Support"]*0.97),
        fill="toself", fillcolor="rgba(0,200,100,0.10)",
        line=dict(color="rgba(0,0,0,0)"), name="Zona Beli",
        showlegend=True, hoverinfo="skip",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=list(dates)+list(dates[::-1]),
        y=list(df["Resistance"]*1.03)+list(df["Resistance"]),
        fill="toself", fillcolor="rgba(220,50,50,0.10)",
        line=dict(color="rgba(0,0,0,0)"), name="Zona Jual",
        showlegend=True, hoverinfo="skip",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=dates, y=close, mode="lines",
        line=dict(color="#4FC3F7", width=2.5, shape="spline", smoothing=0.8),
        name=f"Harga {ticker}",
        hovertemplate="<b>%{x|%d %b %Y}</b><br>Rp %{y:,.0f}<extra></extra>",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=dates, y=df["Support"], mode="lines",
        line=dict(color="#00E676", width=1.0, dash="dot"),
        name="Support Dinamis",
    ), row=1, col=1)

    fig.add_trace(go.Scatter(
        x=dates, y=df["Resistance"], mode="lines",
        line=dict(color="#FF5252", width=1.0, dash="dot"),
        name="Resistance Dinamis",
    ), row=1, col=1)

    x0, x1 = dates.iloc[0], dates.iloc[-1]
    for i, sup in enumerate(sup_levels):
        fig.add_shape(type="line", x0=x0, x1=x1, y0=sup, y1=sup,
                      xref="x", yref="y",
                      line=dict(color="#00FF88", dash="dashdot", width=2.0))
        fig.add_annotation(x=x1, y=sup, xref="x", yref="y",
                           text=f"  S{i+1}: Rp {sup:,.0f}",
                           showarrow=False, font=dict(color="#00FF88", size=10),
                           xanchor="left")

    for i, res in enumerate(res_levels):
        fig.add_shape(type="line", x0=x0, x1=x1, y0=res, y1=res,
                      xref="x", yref="y",
                      line=dict(color="#FF6B6B", dash="dashdot", width=2.0))
        fig.add_annotation(x=x1, y=res, xref="x", yref="y",
                           text=f"  R{i+1}: Rp {res:,.0f}",
                           showarrow=False, font=dict(color="#FF6B6B", size=10),
                           xanchor="left")

    peaks_idx   = cycle.get("peaks_idx",   [])
    valleys_idx = cycle.get("valleys_idx", [])
    if len(peaks_idx) > 0:
        fig.add_trace(go.Scatter(
            x=df["Date"].iloc[peaks_idx], y=df["Close_IDR"].iloc[peaks_idx],
            mode="markers",
            marker=dict(symbol="triangle-down", color="#FF5252", size=12,
                        line=dict(color="white", width=1)),
            name="Puncak Siklus",
        ), row=1, col=1)
    if len(valleys_idx) > 0:
        fig.add_trace(go.Scatter(
            x=df["Date"].iloc[valleys_idx], y=df["Close_IDR"].iloc[valleys_idx],
            mode="markers",
            marker=dict(symbol="triangle-up", color="#00E676", size=12,
                        line=dict(color="white", width=1)),
            name="Lembah Siklus",
        ), row=1, col=1)

    npk = cycle.get("next_peak_date")
    nvl = cycle.get("next_valley_date")
    if npk:
        tambah_vline_manual(fig, npk, "#FF5252", "dash",
                            f"Puncak ~{pd.Timestamp(npk).strftime('%d %b')}")
    if nvl:
        tambah_vline_manual(fig, nvl, "#00E676", "dash",
                            f"Lembah ~{pd.Timestamp(nvl).strftime('%d %b')}")

    colors_vol = [
        "#00E676" if df["Close_IDR"].iloc[i] >= df["Open_IDR"].iloc[i] else "#FF5252"
        for i in range(len(df))
    ]
    fig.add_trace(go.Bar(
        x=dates, y=df["Volume"], marker_color=colors_vol,
        name="Volume", opacity=0.65,
    ), row=2, col=1)

    fig.update_layout(
        template="plotly_dark", height=700,
        title=dict(
            text=f"MAESTRO -- {ticker} | S/R Otomatis | Zona Beli & Jual | Siklus Waktu",
            font=dict(size=14),
        ),
        legend=dict(orientation="h", yanchor="bottom", y=1.01, xanchor="left", x=0),
        hovermode="x unified", dragmode="pan",
        xaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.05)",
                   rangeslider=dict(visible=False), type="date"),
        yaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.07)",
                   tickprefix="Rp ", tickformat=",.0f", side="right"),
        xaxis2=dict(showgrid=False, type="date"),
        yaxis2=dict(showgrid=False),
        margin=dict(l=10, r=130, t=90, b=10),
        paper_bgcolor="#0d1117", plot_bgcolor="#0d1117",
    )

    config = dict(scrollZoom=True, displayModeBar=True,
                  modeBarButtonsToAdd=["drawline", "drawopenpath", "eraseshape"])
    fig.show(config=config)
    cetak_narasi(ticker, df, cycle, result)


def cetak_narasi(ticker, df, cycle, result):
    close   = df["Close_IDR"].iloc[-1]
    support = df["Support"].dropna().iloc[-1] if not df["Support"].dropna().empty else close * 0.95
    resist  = df["Resistance"].dropna().iloc[-1] if not df["Resistance"].dropna().empty else close * 1.05
    pct     = result["pct_change"]
    bandar  = df["Sinyal_Bandarmologi"].iloc[-1] if "Sinyal_Bandarmologi" in df.columns else "[ ] NETRAL"
    tanda   = "+" if pct >= 0 else ""

    npk = cycle.get("next_peak_date")
    nvl = cycle.get("next_valley_date")

    print("\n" + "-" * 70)
    print(f"  NARASI OTOMATIS & TRADING PLAN -- {ticker}")
    print("-" * 70)
    print(f"  Harga {'naik' if pct > 0 else 'turun'} {abs(pct):.2f}% -> Rp {close:,.0f}")
    print(f"  Bandarmologi Live   : {bandar}")

    print("\n  RENCANA EKSEKUSI (TRADING PLAN):")
    print(f"  [BELI]  Area Beli Ideal : Rp {support:,.0f} s/d Rp {support * 1.02:,.0f}")
    print(f"  [BUY+]  Buy Breakout    : Tembus kuat di atas Rp {resist * 1.01:,.0f}")
    print(f"  [TP]    Take Profit     : Rp {resist * 0.98:,.0f}")
    print(f"  [SL]    Cut Loss        : Rp {support * 0.96:,.0f}")
    print(f"  [R/R]   Risk/Reward     : 1 : {((resist*0.98 - support) / (support - support*0.96)):.1f}")

    print("\n  JADWAL TRADING 7 HARI KE DEPAN:")
    sekarang = pd.Timestamp.now().normalize()
    hari_indo = {"Monday": "Senin", "Tuesday": "Selasa", "Wednesday": "Rabu",
                 "Thursday": "Kamis", "Friday": "Jumat", "Saturday": "Sabtu",
                 "Sunday": "Minggu"}

    for i in range(1, 8):
        hari_cek   = sekarang + pd.Timedelta(days=i)
        nama_hari  = hari_indo.get(hari_cek.strftime('%A'), hari_cek.strftime('%A'))
        tgl_str    = f"{nama_hari}, {hari_cek.strftime('%d %b')}"
        aksi       = "[ ] Wait & See"

        if nvl and hari_cek == pd.Timestamp(nvl).normalize():
            aksi = f"[BELI] Prediksi Lembah -> area Rp {support:,.0f}"
        elif npk and hari_cek == pd.Timestamp(npk).normalize():
            aksi = f"[JUAL] Prediksi Puncak -> area Rp {resist:,.0f}"

        print(f"  {tgl_str:<20} : {aksi}")

    print("-" * 70 + "\n")


def cetak_plan_trading_1bulan(ticker: str, cycle: dict, df_signals: pd.DataFrame):
    """
    Cetak plan trading lengkap 1 bulan ke depan berbasis siklus data asli.
    Semua level harga dihitung dari data aktual, bukan hardcode.
    """
    close   = df_signals["Close_IDR"].iloc[-1]
    support = df_signals["Support"].dropna().iloc[-1] if not df_signals["Support"].dropna().empty else close * 0.95
    resist  = df_signals["Resistance"].dropna().iloc[-1] if not df_signals["Resistance"].dropna().empty else close * 1.05
    jadwal  = cycle.get("jadwal_30hari", [])

    print("\n" + "=" * 70)
    print(f"  PLAN TRADING 1 BULAN KE DEPAN -- {ticker}")
    print(f"  (Data berbasis siklus historis aktual, bukan estimasi manual)")
    print("=" * 70)
    print(f"  Harga Saat Ini    : Rp {close:,.0f}")
    print(f"  Area Support Kini : Rp {support:,.0f}")
    print(f"  Area Resist Kini  : Rp {resist:,.0f}")
    print(f"  Avg Siklus Puncak : {cycle.get('avg_peak_cycle', 'N/A')} hari")
    print(f"  Avg Siklus Lembah : {cycle.get('avg_valley_cycle', 'N/A')} hari")

    if not jadwal:
        print("\n  [!] Data siklus belum cukup untuk proyeksi 1 bulan.")
        print("  Saran: gunakan period data lebih panjang (misal: 2y).")
    else:
        print(f"\n  JADWAL EVENT TRADING 30 HARI KE DEPAN ({len(jadwal)} event):")
        print("  " + "-" * 66)
        print(f"  {'No':<4} {'Tanggal':<18} {'Hari':<10} {'Event':<20} {'Aksi Yang Disarankan'}")
        print("  " + "-" * 66)

        hari_indo = {"Monday": "Senin", "Tuesday": "Selasa", "Wednesday": "Rabu",
                     "Thursday": "Kamis", "Friday": "Jumat", "Saturday": "Sabtu",
                     "Sunday": "Minggu"}

        for idx, item in enumerate(jadwal, 1):
            tgl      = item["tanggal"]
            nm_hari  = hari_indo.get(tgl.strftime('%A'), tgl.strftime('%A'))
            tgl_str  = tgl.strftime('%d %b %Y')
            event    = item["event"]
            aksi     = item["aksi"]

            if item["type"] == "valley":
                harga_acuan = f"Target beli area Rp {support:,.0f}"
            else:
                harga_acuan = f"Target jual area Rp {resist:,.0f}"

            print(f"  {idx:<4} {tgl_str:<18} {nm_hari:<10} {event:<20} {aksi}")
            print(f"  {'':4} {'':<18} {'':<10} {'':20} -> {harga_acuan}")

        print("  " + "-" * 66)
        print("\n  STRATEGI UMUM 1 BULAN:")
        print(f"  - Modal per entry   : Sesuaikan dengan risk 2% dari total portofolio")
        print(f"  - Stop Loss global  : Rp {support * 0.95:,.0f} (di bawah support 5%)")
        print(f"  - Target konservatif: Rp {resist * 0.97:,.0f}")
        print(f"  - Target agresif    : Rp {resist * 1.03:,.0f}")
        print("  - Review mingguan   : Pantau volume spike sebagai konfirmasi")

    print("=" * 70 + "\n")


print("[OK] Cell 6 - Chart TradingView + Plan Trading 1 Bulan berhasil dimuat.")

[OK] Cell 6 - Chart TradingView + Plan Trading 1 Bulan berhasil dimuat.


---
## Cell 7 - Engine Berita & Analisis Sentimen (Indonesia)

**Fitur Baru v3:**
- Scraping berita Indonesia dari sumber publik (Bisnis.com, Kontan, IDX, Google News)
- Analisis sentimen kata kunci Bahasa Indonesia + Inggris
- Berpura-pura sebagai browser biasa (User-Agent spoofing) untuk akses publik
- Fallback ke yfinance.news jika scraping gagal

In [9]:
# ======================================================================
# CELL 7: ENGINE BERITA & ANALISIS SENTIMEN INDONESIA
# Menggunakan scraping publik + yfinance.news + keyword analysis
# ======================================================================

KATA_POSITIF = [
    "naik", "untung", "laba", "profit", "growth", "bullish", "beli", "buy",
    "rekomendasi", "upgrade", "outperform", "strong", "rally", "record",
    "gain", "rise", "up", "beat", "exceed", "positive", "optimis", "boom",
    "akuisisi", "merger", "dividen", "buyback", "ekspansi", "inovasi",
    "kuat", "meningkat", "surplus", "melebihi", "ekspektasi", "rebound",
    "recover", "breakout", "all time high", "ath", "tumbuh", "berkembang",
    "prospek cerah", "solid", "kinerja baik", "pendapatan naik"
]

KATA_NEGATIF = [
    "turun", "rugi", "loss", "bearish", "jual", "sell", "downgrade",
    "underperform", "weak", "crash", "drop", "fall", "down", "miss",
    "negative", "pesimis", "bangkrut", "delisting", "korupsi", "skandal",
    "gugatan", "sanksi", "pailit", "defisit", "resesi", "melemah",
    "merosot", "anjlok", "tertekan", "jatuh", "gagal", "kerugian",
    "hutang", "default", "kolaps", "inflasi tinggi", "pemangkasan"
]

# Header browser palsu agar tidak diblokir sebagai bot
BROWSER_HEADERS = [
    {
        "User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
                      "AppleWebKit/537.36 (KHTML, like Gecko) "
                      "Chrome/122.0.0.0 Safari/537.36",
        "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8,en;q=0.7",
        "Accept": "text/html,application/xhtml+xml,application/xml;q=0.9,*/*;q=0.8",
        "Referer": "https://www.google.co.id/",
    },
    {
        "User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
                      "AppleWebKit/605.1.15 (KHTML, like Gecko) "
                      "Version/17.3 Safari/605.1.15",
        "Accept-Language": "id-ID,id;q=0.9",
    },
]


def analisis_sentimen_teks(teks: str) -> tuple:
    """Analisis sentimen dari teks. Return (label, skor)."""
    teks_lower = teks.lower()
    pos = sum(1 for k in KATA_POSITIF if k in teks_lower)
    neg = sum(1 for k in KATA_NEGATIF if k in teks_lower)
    if pos > neg:   return "[+] POSITIF", pos - neg
    elif neg > pos: return "[-] NEGATIF", neg - pos
    else:           return "[=] NETRAL",  0


def bersihkan_ticker_untuk_cari(ticker: str) -> str:
    """Konversi ticker ke kata kunci pencarian yang lebih manusiawi."""
    ticker = ticker.upper()
    peta = {
        "BBCA.JK": "Bank BCA BBCA", "BBRI.JK": "Bank BRI BBRI",
        "BMRI.JK": "Bank Mandiri",   "TLKM.JK": "Telkom Indonesia",
        "GOTO.JK": "GoTo Gojek",     "ASII.JK": "Astra Internasional",
        "BTC-USD":  "Bitcoin harga", "ETH-USD":  "Ethereum harga",
        "EURUSD=X": "Euro Dollar",   "XAUUSD=X": "Emas harga",
        "AAPL":    "Apple Inc",      "NVDA":     "Nvidia saham",
    }
    if ticker in peta:
        return peta[ticker]
    t = ticker.replace(".JK", "").replace("-USD", "").replace("=X", "")
    return t


def scrape_google_news_indonesia(query: str, max_hasil: int = 5) -> list:
    """
    Scrape Google News (bahasa Indonesia) seperti manusia biasa.
    Akses konten publik yang terbuka untuk umum.
    """
    hasil = []
    try:
        q_encoded = quote(f"{query} saham investasi")
        url = f"https://news.google.com/rss/search?q={q_encoded}&hl=id&gl=ID&ceid=ID:id"
        headers = random.choice(BROWSER_HEADERS)
        resp = requests.get(url, headers=headers, timeout=8)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "lxml-xml")
            items = soup.find_all("item")
            for item in items[:max_hasil]:
                judul = item.find("title")
                link  = item.find("link")
                tgl   = item.find("pubDate")
                if judul:
                    hasil.append({
                        "judul": judul.text.strip(),
                        "link":  link.text.strip() if link else "",
                        "tgl":   tgl.text[:16] if tgl else "Baru",
                        "sumber": "Google News ID",
                    })
    except Exception:
        pass
    return hasil


def scrape_investing_id(ticker_kode: str, max_hasil: int = 3) -> list:
    """
    Scrape berita dari Investing.com versi Indonesia.
    """
    hasil = []
    try:
        kode = ticker_kode.replace(".JK", "").replace("-USD", "").replace("=X", "").lower()
        url  = f"https://id.investing.com/search/?q={kode}&tab=news"
        headers = random.choice(BROWSER_HEADERS)
        resp = requests.get(url, headers=headers, timeout=8)
        if resp.status_code == 200:
            soup = BeautifulSoup(resp.text, "html.parser")
            articles = soup.find_all(["article", "div"],
                                     class_=re.compile(r"articleItem|news-item|article"), limit=5)
            for art in articles[:max_hasil]:
                title_tag = art.find(["h3", "h4", "a", "span"])
                if title_tag and len(title_tag.text.strip()) > 10:
                    hasil.append({
                        "judul":  title_tag.text.strip()[:150],
                        "link":   "",
                        "tgl":    "Terkini",
                        "sumber": "Investing.com ID",
                    })
    except Exception:
        pass
    return hasil


def ambil_berita_yfinance(ticker: str, max_berita: int = 5) -> list:
    """Ambil berita dari yfinance sebagai fallback."""
    hasil = []
    try:
        t    = yf.Ticker(ticker)
        news = t.news
        if not news:
            return hasil
        for item in news[:max_berita]:
            content = item.get("content", {})
            if isinstance(content, dict):
                judul  = content.get("title", "")
                sumber = content.get("provider", {}).get("displayName", "Yahoo Finance")
                tgl    = content.get("pubDate", "")[:16]
            else:
                judul  = item.get("title", "")
                sumber = item.get("publisher", "Yahoo Finance")
                tgl    = ""
            if judul:
                hasil.append({
                    "judul":  judul[:150],
                    "link":   "",
                    "tgl":    tgl,
                    "sumber": sumber,
                })
    except Exception:
        pass
    return hasil


def cetak_berita_dan_sentimen(ticker: str, is_jk: bool = False, max_berita: int = 8) -> str:
    """
    Tampilkan berita & analisis sentimen dari berbagai sumber.
    Untuk saham Indonesia (.JK): prioritas ke Google News Bahasa Indonesia.
    Return: sentimen keseluruhan.
    """
    print(f"\n{'='*65}")
    print(f"  BERITA & SENTIMEN PASAR -- {ticker}")
    print(f"{'='*65}")

    semua_berita = []
    query_indo   = bersihkan_ticker_untuk_cari(ticker)

    # Sumber 1: Google News Indonesia (prioritas utama untuk .JK)
    print("  [>] Mencari berita Indonesia (Google News)...")
    berita_google = scrape_google_news_indonesia(query_indo, max_hasil=5)
    semua_berita.extend(berita_google)

    # Sumber 2: Investing.com ID
    print("  [>] Mencari di Investing.com Indonesia...")
    berita_inv = scrape_investing_id(ticker, max_hasil=3)
    semua_berita.extend(berita_inv)

    # Sumber 3: yfinance.news sebagai fallback
    if len(semua_berita) < 3:
        print("  [>] Fallback ke Yahoo Finance News...")
        berita_yf = ambil_berita_yfinance(ticker, max_berita=5)
        semua_berita.extend(berita_yf)

    # --- Cetak berita ---
    if not semua_berita:
        print("  [!] Tidak ada berita ditemukan saat ini.")
        print(f"{'='*65}\n")
        return "[=] NETRAL"

    print(f"\n  Ditemukan {len(semua_berita)} berita. Menampilkan {min(max_berita, len(semua_berita))}:")
    print("-" * 65)

    skor_kum   = 0
    n_positif  = 0
    n_negatif  = 0
    n_netral   = 0

    for i, b in enumerate(semua_berita[:max_berita], 1):
        judul   = b.get("judul", "")
        sumber  = b.get("sumber", "")
        tgl     = b.get("tgl", "")
        sentimen, skor = analisis_sentimen_teks(judul)

        skor_kum += (skor if sentimen.startswith("[+") else -skor)
        if sentimen.startswith("[+"): n_positif += 1
        elif sentimen.startswith("[-"): n_negatif += 1
        else: n_netral += 1

        print(f"  [{i:2d}] {sentimen}  |  {tgl[:10]}  |  {sumber}")
        print(f"       {judul[:90]}")
        if i < min(max_berita, len(semua_berita)):
            print()

    # --- Ringkasan sentimen ---
    print("-" * 65)
    print(f"  Positif: {n_positif}  |  Negatif: {n_negatif}  |  Netral: {n_netral}")

    if skor_kum > 0:   overall = "[+] BULLISH (Sentimen Positif)"
    elif skor_kum < 0: overall = "[-] BEARISH (Sentimen Negatif)"
    else:              overall = "[=] NETRAL"

    print(f"  Sentimen Keseluruhan: {overall}")
    print(f"{'='*65}\n")
    return overall


print("[OK] Cell 7 - Engine Berita & Sentimen Indonesia berhasil dimuat.")

[OK] Cell 7 - Engine Berita & Sentimen Indonesia berhasil dimuat.


---
## Cell 8A - ANALISIS 1 ASET (Input Keyboard)

> Semua aset didukung: saham IDX, crypto, forex, saham US

In [10]:
# ======================================================================
# CELL 8A: ANALISIS MENDALAM 1 ASET -- INPUT VIA KEYBOARD
# Fix v3: Support semua jenis aset tanpa error
# ======================================================================

print("=" * 65)
print("  MAESTRO -- ANALISIS 1 ASET (Input Keyboard)")
print("=" * 65)
print("  Contoh ticker:")
print("  Saham IDX : BBCA.JK  TLKM.JK  GOTO.JK  BBRI.JK")
print("  Crypto    : BTC-USD   ETH-USD   BNB-USD   SOL-USD")
print("  Forex     : EURUSD=X  USDJPY=X  XAUUSD=X")
print("  Saham US  : AAPL      NVDA      MSFT      TSLA")
print("=" * 65)

ticker_input  = input("\n  Masukkan Ticker Aset  : ").strip().upper()
periode_input = input("  Periode Data [default: 1y] : ").strip() or "1y"

if not ticker_input:
    print("  [!] Ticker tidak boleh kosong!")
else:
    print(f"\n  Menganalisis: {ticker_input} | Periode: {periode_input}\n")
    try:
        result = tarik_data_maestro(ticker_input, period=periode_input)
        df     = result["df"]
        is_jk  = result["is_jk"]
        ticker = result["ticker"]

        # 1. Analisis Keuangan Lengkap
        fund = ambil_fundamental(ticker)
        cetak_analisis_keuangan_lengkap(ticker, fund, is_jk)

        # 2. Berita & Sentimen
        sentimen_overall = cetak_berita_dan_sentimen(ticker, is_jk)

        # 3. Siklus Waktu
        cycle = deteksi_siklus_waktu(df, prominence_pct=0.015)
        print_cycle_summary(ticker, cycle)

        # 4. Sinyal Trading
        df["Support"]    = df["Low_IDR"].rolling(20).min().shift(1)
        df["Resistance"] = df["High_IDR"].rolling(20).max().shift(1)
        df_signals = deteksi_sinyal_semua(cycle["df_annotated"], cycle)
        df_signals["Support"]    = df["Support"]
        df_signals["Resistance"] = df["Resistance"]
        ringkasan_sinyal(df_signals, ticker)

        # 5. Plan Trading 1 Bulan
        cetak_plan_trading_1bulan(ticker, cycle, df_signals)

        # 6. Chart Interaktif
        buat_chart_maestro(result, cycle, df_signals)

        # 7. Kesimpulan Akhir
        skor_fund = hitung_skor_fundamental(fund)
        print("\n" + "=" * 65)
        print(f"  KESIMPULAN MAESTRO -- {ticker}")
        print("=" * 65)
        print(f"  Harga Terakhir         : Rp {result['harga_terakhir_idr']:,.0f}")
        print(f"  Perubahan Hari Ini     : {'+' if result['pct_change'] >= 0 else ''}{result['pct_change']:.2f}%")
        print(f"  Skor Fundamental       : {skor_fund}/100")
        print(f"  Sentimen Pasar         : {sentimen_overall}")
        print(f"  Puncak Berikutnya      : {cycle['next_peak_date'].strftime('%d %b %Y') if cycle['next_peak_date'] else 'N/A'}")
        print(f"  Lembah Berikutnya      : {cycle['next_valley_date'].strftime('%d %b %Y') if cycle['next_valley_date'] else 'N/A'}")
        if is_jk:
            print(f"  Harga 1 Lot (100 lbr)  : Rp {result['harga_terakhir_idr']*100:,.0f}")
        print("=" * 65)

    except Exception as e:
        print(f"\n  [ERROR] {e}")
        import traceback
        traceback.print_exc()

  MAESTRO -- ANALISIS 1 ASET (Input Keyboard)
  Contoh ticker:
  Saham IDX : BBCA.JK  TLKM.JK  GOTO.JK  BBRI.JK
  Crypto    : BTC-USD   ETH-USD   BNB-USD   SOL-USD
  Forex     : EURUSD=X  USDJPY=X  XAUUSD=X
  Saham US  : AAPL      NVDA      MSFT      TSLA

  Menganalisis: DEWA.JK | Periode: 5mo

  [IDX]    [DEWA.JK       ]  Rp           486.00  (-5.63%)  -  IDR (langsung)

  ANALISIS KEUANGAN LENGKAP -- DEWA.JK
  [CO] PT Darma Henwa Tbk
  [SE] Sektor   : Energy
  [IN] Industri : Thermal Coal
  [NB] Profil   : PT Darma Henwa Tbk, together with its subsidiaries, engages in the mining support and other excavation activities, and metal fabrication products in Indonesia. It operates in two segments, Mining Serv...

  -- VALUASI ----------------------------------------------------
  PER (P/E Ratio)    : 4.3x  [Murah]
  PBV (Price/Book)   : 2.30x  [Wajar]

  -- PROFITABILITAS ---------------------------------------------
  ROE                : 72.4%  [Sangat Baik]
  Net Profit Margin  : 67.4%

Price,Date,Close_IDR,Sinyal_Bandarmologi,Sinyal_Scalping,Sinyal_Swing,Sinyal_BPJS,Sinyal_BSJP
0,20 Apr 2026,Rp 550,[ ] NETRAL,--,--,--,--
1,21 Apr 2026,Rp 560,[ ] NETRAL,--,--,>> BPJS,--
2,22 Apr 2026,Rp 550,[ ] NETRAL,--,--,--,--
3,23 Apr 2026,Rp 515,[ ] NETRAL,--,--,--,--
4,24 Apr 2026,Rp 486,[ ] NETRAL,--,--,--,--



  PLAN TRADING 1 BULAN KE DEPAN -- DEWA.JK
  (Data berbasis siklus historis aktual, bukan estimasi manual)
  Harga Saat Ini    : Rp 486
  Area Support Kini : Rp 432
  Area Resist Kini  : Rp 595
  Avg Siklus Puncak : 12 hari
  Avg Siklus Lembah : 13 hari

  JADWAL EVENT TRADING 30 HARI KE DEPAN (4 event):
  ------------------------------------------------------------------
  No   Tanggal            Hari       Event                Aksi Yang Disarankan
  ------------------------------------------------------------------
  1    30 Apr 2026        Kamis      LEMBAH SIKLUS        BELI / AKUMULASI
                                                          -> Target beli area Rp 432
  2    03 May 2026        Minggu     PUNCAK SIKLUS        JUAL / TAKE PROFIT
                                                          -> Target jual area Rp 595
  3    13 May 2026        Rabu       LEMBAH SIKLUS        BELI / AKUMULASI
                                                          -> Target beli area R


----------------------------------------------------------------------
  NARASI OTOMATIS & TRADING PLAN -- DEWA.JK
----------------------------------------------------------------------
  Harga turun 5.63% -> Rp 486
  Bandarmologi Live   : [ ] NETRAL

  RENCANA EKSEKUSI (TRADING PLAN):
  [BELI]  Area Beli Ideal : Rp 432 s/d Rp 441
  [BUY+]  Buy Breakout    : Tembus kuat di atas Rp 601
  [TP]    Take Profit     : Rp 583
  [SL]    Cut Loss        : Rp 415
  [R/R]   Risk/Reward     : 1 : 8.7

  JADWAL TRADING 7 HARI KE DEPAN:
  Senin, 27 Apr        : [ ] Wait & See
  Selasa, 28 Apr       : [ ] Wait & See
  Rabu, 29 Apr         : [ ] Wait & See
  Kamis, 30 Apr        : [BELI] Prediksi Lembah -> area Rp 432
  Jumat, 01 May        : [ ] Wait & See
  Sabtu, 02 May        : [ ] Wait & See
  Minggu, 03 May       : [JUAL] Prediksi Puncak -> area Rp 595
----------------------------------------------------------------------


  KESIMPULAN MAESTRO -- DEWA.JK
  Harga Terakhir         : Rp 486
  Per

---
## Cell 8B - SCREENER SAHAM IDX

> Hanya input budget, sistem scan otomatis saham IDX

In [ ]:
# ======================================================================
# CELL 8B: MEGA SCREENER SAHAM IDX
# ======================================================================

def ambil_ticker_otomatis():
    print("  [>] Mengakses database saham Indonesia...")
    try:
        url     = "https://id.wikipedia.org/wiki/Indeks_Kompas100"
        headers = random.choice(BROWSER_HEADERS)
        resp    = requests.get(url, headers=headers, timeout=10)
        tables  = pd.read_html(resp.text)

        for tbl in tables:
            kolom_kode = [col for col in tbl.columns
                          if any(k in str(col) for k in ['Kode', 'Ticker', 'Simbol'])]
            if kolom_kode:
                emiten  = tbl[kolom_kode[0]].dropna().astype(str).tolist()
                emiten  = [e.strip() for e in emiten if len(e.strip()) == 4]
                daftar  = [f"{e}.JK" for e in emiten]
                if len(daftar) > 10:
                    print(f"  [OK] Berhasil ambil {len(daftar)} saham Kompas100 dari Wikipedia.")
                    return daftar
        raise ValueError("Struktur tabel tidak sesuai.")
    except Exception as e:
        print(f"  [!] Scraping gagal ({str(e)[:40]}). Pakai LQ45 cadangan.")
        return [
            # ── BLUECHIP / LQ45 ──────────────────────────────────────────────
            "BBCA.JK", "BBRI.JK", "BMRI.JK", "BBNI.JK", "TLKM.JK",
            "ASII.JK", "GOTO.JK", "AMMN.JK", "BREN.JK", "BRIS.JK",
            "UNVR.JK", "ICBP.JK", "PGAS.JK", "PTBA.JK", "ADRO.JK",
            "TPIA.JK", "MEDC.JK", "BRPT.JK", "KLBF.JK", "AKRA.JK",
            "AMRT.JK", "ESSA.JK", "HRUM.JK", "ITMG.JK", "MDKA.JK",
            "PGEO.JK", "SIDO.JK", "SRTG.JK", "BUKA.JK", "CPIN.JK",
            "EXCL.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK",
            "MBMA.JK", "MTEL.JK", "MYOR.JK", "SMGR.JK", "TOWR.JK",
            "ARTO.JK", "SCMA.JK", "EMTK.JK", "ACES.JK", "INCO.JK",

            # ── PERBANKAN & KEUANGAN ─────────────────────────────────────────
            "BTPS.JK", "BJTM.JK", "BJBR.JK", "BNGA.JK", "BNLI.JK",
            "BDMN.JK", "BKSW.JK", "NISP.JK", "MEGA.JK", "AGRO.JK",
            "BBYB.JK", "BNBA.JK", "BMAS.JK", "DNAR.JK", "NAGA.JK",
            "NOBU.JK", "PNBS.JK", "RELI.JK", "SDRA.JK", "BANK.JK",

            # ── ASURANSI & MULTIFINANCE ──────────────────────────────────────
            "WOMF.JK", "BFIN.JK", "MFIN.JK", "VRNA.JK", "TRIM.JK",
            "PANS.JK", "ABMM.JK", "ADMF.JK", "CFIN.JK", "IMJS.JK",

            # ── TEKNOLOGI & DIGITAL ──────────────────────────────────────────
            "DMMX.JK", "FREN.JK", "LUCK.JK", "MIKA.JK", "MTDL.JK",
            "MLPT.JK", "ATIC.JK", "EDGE.JK", "MSTI.JK", "MSIN.JK",
            "AWAN.JK", "KOTA.JK", "AXIO.JK", "SONA.JK", "JADI.JK",

            # ── CONSUMER GOODS & RETAIL ──────────────────────────────────────
            "GGRM.JK", "HMSP.JK", "INDF.JK", "MAPI.JK", "RALS.JK",
            "HERO.JK", "MIDI.JK", "CSAP.JK", "ARGO.JK", "SSTM.JK",
            "AISA.JK", "CAMP.JK", "CEKA.JK", "DLTA.JK", "FOOD.JK",
            "GOOD.JK", "HOKI.JK", "KEJU.JK", "SKBM.JK", "ULTJ.JK",
            "BOBA.JK", "ALTO.JK", "TBLA.JK", "BUDI.JK", "MGNA.JK",

            # ── KESEHATAN & FARMASI ──────────────────────────────────────────
            "KAEF.JK", "MERK.JK", "PYFA.JK", "SIDO.JK", "TSPC.JK",
            "DVLA.JK", "INAF.JK", "HEAL.JK", "MIKA.JK", "SILO.JK",
            "PRDA.JK", "SAME.JK", "BMHS.JK", "OMED.JK", "RSGK.JK",

            # ── PROPERTI & KONSTRUKSI ────────────────────────────────────────
            "BSDE.JK", "CTRA.JK", "LPKR.JK", "PWON.JK", "SMRA.JK",
            "APLN.JK", "ARMY.JK", "ASRI.JK", "BKSL.JK", "CAKK.JK",
            "DILD.JK", "ELTY.JK", "FORZ.JK", "GMTD.JK", "GPRA.JK",
            "JRPT.JK", "KIJA.JK", "LPCK.JK", "MDLN.JK", "MKPI.JK",
            "MTLA.JK", "MMLP.JK", "NIRO.JK", "PLIN.JK", "PPRO.JK",
            "RDTX.JK", "RODA.JK", "SMDM.JK", "TARA.JK", "WIKA.JK",
            "WSKT.JK", "ADHI.JK", "PTPP.JK", "TOTL.JK", "ACST.JK",

            # ── ENERGI & PERTAMBANGAN ────────────────────────────────────────
            "ANTM.JK", "ITMG.JK", "BUMI.JK", "ENRG.JK", "DEWA.JK",
            "GEMS.JK", "KKGI.JK", "MYOH.JK", "PKPK.JK", "SMMT.JK",
            "TOBA.JK", "DOID.JK", "BOSS.JK", "MBAP.JK", "FIRE.JK",
            "GTBO.JK", "APEX.JK", "SMRU.JK", "ELSA.JK", "RUIS.JK",
            "ARTI.JK", "BIPI.JK", "LSIP.JK", "SIMP.JK", "TAPG.JK",

            # ── MINYAK SAWIT & AGRIBISNIS ────────────────────────────────────
            "AALI.JK", "BWPT.JK", "DSNG.JK", "GZCO.JK", "JAWA.JK",
            "PALM.JK", "SGRO.JK", "SMAR.JK", "SSMS.JK", "TGRA.JK",
            "MAGP.JK", "ANJT.JK", "BNRS.JK", "GOLL.JK", "INDO.JK",

            # ── INFRASTRUKTUR & UTILITAS ─────────────────────────────────────
            "JSMR.JK", "WTON.JK", "WEGE.JK", "META.JK", "TBIG.JK",
            "SUPR.JK", "KPIG.JK", "IPCC.JK", "TLKM.JK", "IPCM.JK",
            "BIRD.JK", "BLTZ.JK", "GIAA.JK", "NELY.JK", "SAFE.JK",

            # ── MANUFAKTUR & INDUSTRI ────────────────────────────────────────
            "AUTO.JK", "GJTL.JK", "INDS.JK", "NIPS.JK", "PRAS.JK",
            "SMSM.JK", "ADMG.JK", "IMAS.JK", "LPIN.JK", "MASA.JK",
            "SRIL.JK", "RICY.JK", "POLY.JK", "ERTX.JK", "ARGO.JK",
            "BATA.JK", "IKBI.JK", "KBLM.JK", "SCCO.JK", "VOKS.JK",
            "ALKA.JK", "ALMI.JK", "BTON.JK", "GDST.JK", "ISSP.JK",
            "JKSW.JK", "KRAS.JK", "LION.JK", "TBMS.JK", "ZINC.JK",

            # ── SEMEN & MATERIAL ─────────────────────────────────────────────
            "SMBR.JK", "JECC.JK", "ARNA.JK", "MARK.JK", "MLIA.JK",
            "TOTO.JK", "KIAS.JK", "AMFG.JK", "CANGK.JK","IFII.JK",

            # ── HIDDEN GEM — KAPITALISASI KECIL-MENENGAH ────────────────────
            "OILS.JK", "ANDI.JK", "CUAN.JK", "PNLF.JK", "LPGI.JK",
            "YELO.JK", "UCID.JK", "UFOE.JK", "MSKY.JK", "VICI.JK",
            "SWAT.JK", "GAMA.JK", "LABA.JK", "BHAT.JK", "BPTR.JK",
            "RCCC.JK", "MNCN.JK", "KBLV.JK", "PBSA.JK", "HATM.JK",
            "MABA.JK", "GRIA.JK", "BPII.JK", "HILL.JK", "TGUK.JK",
            "PMMP.JK", "DMAS.JK", "URBN.JK", "NASI.JK", "RAFI.JK",

            # ── MEDIA & ENTERTAINMENT ────────────────────────────────────────
            "FILM.JK", "IPTV.JK", "KREN.JK", "LINK.JK", "MDIA.JK",
            "MSIN.JK", "NFCX.JK", "CENT.JK", "ABBA.JK", "RIMO.JK",
        ]


print("=" * 65)
print("  MAESTRO -- MEGA SCREENER SAHAM IDX")
print("  Scan otomatis saham terbaik sesuai budget Anda")
print("=" * 65)

try:
    budget_str   = input("\n  Budget Anda (Rupiah) [contoh: 500000] : ").strip()
    budget_saham = float(budget_str.replace(",", "").replace(".", ""))
except ValueError:
    budget_saham = 500_000
    print(f"  [!] Menggunakan default: Rp {budget_saham:,.0f}")

DAFTAR_SAHAM = ambil_ticker_otomatis()
limit_scan   = 45

print(f"\n  Budget   : Rp {budget_saham:,.0f}")
print(f"  Scanning : {min(limit_scan, len(DAFTAR_SAHAM))} saham (mohon tunggu...)\n")

data_saham  = tarik_data_batch(DAFTAR_SAHAM[:limit_scan], period="6mo")
tabel_saham = screener_budget(data_saham, budget=budget_saham)

print("\n" + "=" * 65)
print("  HASIL MEGA SCREENER SAHAM IDX")
print("=" * 65)
display(tabel_saham)

lolos = tabel_saham[tabel_saham["Budget OK?"] == "[v]"]
if not lolos.empty:
    top = lolos.iloc[0]
    print(f"\n  Saham Terbaik dalam Budget Rp {budget_saham:,.0f}:")
    print(f"  -> {top['Ticker']} | {top['Nama']}")
    print(f"     Harga 1 Lot: Rp {top['Min. Beli (IDR)']:,.0f} | Skor: {top['Skor (0-100)']}/100")
else:
    print(f"\n  [!] Tidak ada saham yang lolos budget Rp {budget_saham:,.0f}.")
    print("  Saran: Budget minimum Bluechip biasanya Rp 300.000 - 600.000.")

  MAESTRO -- MEGA SCREENER SAHAM IDX
  Scan otomatis saham terbaik sesuai budget Anda
  [!] Menggunakan default: Rp 500,000
  [>] Mengakses database saham Indonesia...
  [!] Scraping gagal ([Errno 2] No such file or directory: <!D). Pakai LQ45 cadangan.

  Budget   : Rp 500,000
  Scanning : 45 saham (mohon tunggu...)


  MAESTRO DATA ENGINE - MENARIK DATA MULTI-ASET
  [IDX]    [BBCA.JK       ]  Rp         6,050.00  (-5.84%)  -  IDR (langsung)
  [IDX]    [BBRI.JK       ]  Rp         3,070.00  (-2.85%)  -  IDR (langsung)
  [IDX]    [BMRI.JK       ]  Rp         4,500.00  (-2.81%)  -  IDR (langsung)
  [IDX]    [BBNI.JK       ]  Rp         3,770.00  (-2.58%)  -  IDR (langsung)
  [IDX]    [TLKM.JK       ]  Rp         2,810.00  (-2.43%)  -  IDR (langsung)
  [IDX]    [ASII.JK       ]  Rp         6,325.00  (+0.00%)  -  IDR (langsung)
  [IDX]    [GOTO.JK       ]  Rp            52.00  (-3.70%)  -  IDR (langsung)
  [IDX]    [AMMN.JK       ]  Rp         5,000.00  (-4.31%)  -  IDR (langsung)
  [IDX]

,Ticker,Nama,Sektor,Harga (IDR),Min. Beli (IDR),Budget OK?,PER,PBV,ROE (%),Skor (0-100)
1,EMTK.JK,PT Elang Mahkota Teknologi Tbk,Communication Services,850.0,85000.0,[v],7.6,1.37,19.1%,92.0
2,BUKA.JK,PT Bukalapak.com Tbk.,Consumer Cyclical,159.0,15900.0,[v],5.1,0.65,12.8%,89.0
3,SRTG.JK,PT Saratoga Investama Sedaya T,Financial Services,1755.0,175500.0,[v],3.3,0.40,13.2%,89.0
4,SIDO.JK,PT Industri Jamu dan Farmasi S,Healthcare,498.0,49800.0,[v],12.0,4.70,37.2%,83.0
5,KLBF.JK,PT Kalbe Farma Tbk.,Healthcare,875.0,87500.0,[v],10.9,1.70,15.2%,82.0
6,BMRI.JK,PT Bank Mandiri (Persero) Tbk,Financial Services,4500.0,450000.0,[v],7.5,1.38,21.0%,82.0
7,BBRI.JK,PT Bank Rakyat Indonesia (Pers,Financial Services,3070.0,307000.0,[v],8.2,1.43,17.5%,82.0
8,BBNI.JK,PT Bank Negara Indonesia (Pers,Financial Services,3770.0,377000.0,[v],7.0,0.82,11.7%,79.0
9,BRIS.JK,PT Bank Syariah Indonesia Tbk,Financial Services,1880.0,188000.0,[v],11.5,1.67,15.6%,79.0
10,CPIN.JK,PT Charoen Pokphand Indonesia,Consumer Defensive,4140.0,414000.0,[v],12.0,1.99,17.5%,78.0



  Saham Terbaik dalam Budget Rp 500,000:
  -> EMTK.JK | PT Elang Mahkota Teknologi Tbk
     Harga 1 Lot: Rp 85,000 | Skor: 92.0/100


---
## Cell 8C - SCREENER CRYPTOCURRENCY

> Hanya input budget, sistem scan otomatis top crypto

In [ ]:
# ======================================================================
# CELL 8C: SCREENER CRYPTO -- Fix v3: tidak lagi error
# ======================================================================

DAFTAR_CRYPTO = [
    # ── LARGE CAP ───────────────────────────────────────
    "BTC-USD", "ETH-USD", "BNB-USD", "SOL-USD", "XRP-USD",
    "ADA-USD", "AVAX-USD", "DOGE-USD", "TRX-USD", "TON-USD",
    # ── MID CAP POPULER ─────────────────────────────────
    "LINK-USD", "DOT-USD", "MATIC-USD", "LTC-USD", "ATOM-USD",
    "UNI-USD", "NEAR-USD", "APT-USD", "OP-USD", "ARB-USD",
    "IMX-USD", "INJ-USD", "SUI-USD", "SEI-USD", "JUP-USD",
    # ── HIDDEN GEM POTENSIAL ─────────────────────────────
    "FET-USD", "RENDER-USD", "WLD-USD", "ONDO-USD", "PENDLE-USD",
    "STRK-USD", "PYTH-USD", "JTO-USD", "ETHFI-USD", "EIGEN-USD",
    "MANTA-USD", "ALT-USD", "PORTAL-USD", "PIXEL-USD", "SAFE-USD",
    # ── DEFI & UTILITY ──────────────────────────────────
    "AAVE-USD", "MKR-USD", "CRV-USD", "LDO-USD", "RPL-USD",
    "SNX-USD", "BAL-USD", "COMP-USD", "SUSHI-USD", "1INCH-USD",
    # ── KOMODITAS CRYPTO ────────────────────────────────
    "PAXG-USD",
]

print("=" * 65)
print("  MAESTRO -- SCREENER CRYPTOCURRENCY")
print("  Scan otomatis 20 crypto terpopuler")
print("=" * 65)

try:
    budget_str    = input("\n  Budget Anda (Rupiah) [contoh: 1000000] : ").strip()
    budget_crypto = float(budget_str.replace(",", "").replace(".", ""))
except ValueError:
    budget_crypto = 1_000_000
    print(f"  [!] Menggunakan default: Rp {budget_crypto:,.0f}")

print(f"\n  Budget   : Rp {budget_crypto:,.0f}")
print(f"  Scanning : {len(DAFTAR_CRYPTO)} crypto...\n")

data_crypto  = tarik_data_batch(DAFTAR_CRYPTO, period="6mo")
tabel_crypto = screener_budget(data_crypto, budget=budget_crypto)

print("\n" + "=" * 65)
print("  HASIL SCREENER CRYPTO")
print("=" * 65)
display(tabel_crypto)

lolos_c = tabel_crypto[tabel_crypto["Budget OK?"] == "[v]"]
if not lolos_c.empty:
    top_c = lolos_c.iloc[0]
    print(f"\n  Crypto Terjangkau Terbaik:")
    print(f"  -> {top_c['Ticker']} | Harga: Rp {top_c['Harga (IDR)']:,.2f}/unit")
    print(f"     Skor: {top_c['Skor (0-100)']}/100")
else:
    print(f"\n  [!] Semua crypto melebihi budget Rp {budget_crypto:,.0f}")
    print("  Saran: Coba DOGE-USD atau SHIB-USD (harga lebih murah per unit).")

  MAESTRO -- SCREENER CRYPTOCURRENCY
  Scan otomatis 20 crypto terpopuler
  [!] Menggunakan default: Rp 1,000,000

  Budget   : Rp 1,000,000
  Scanning : 20 crypto...


  MAESTRO DATA ENGINE - MENARIK DATA MULTI-ASET
  [CRYPTO] [BTC-USD       ]  Rp 1,344,773,491.45  (+0.31%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [ETH-USD       ]  Rp    40,251,791.44  (+0.50%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [BNB-USD       ]  Rp    10,905,653.93  (+0.43%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [SOL-USD       ]  Rp     1,486,859.87  (-0.08%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [XRP-USD       ]  Rp        24,617.48  (+0.07%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [ADA-USD       ]  Rp         4,366.61  (+1.01%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [AVAX-USD      ]  Rp       163,057.11  (+0.79%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [DOGE-USD      ]  Rp         1,702.95  (+0.55%)  -  IDR (kurs: Rp 17,273)


$MATIC-USD: possibly delisted; no price data found  (period=6mo)

1 Failed download:
['MATIC-USD']: possibly delisted; no price data found  (period=6mo)


  [CRYPTO] [LINK-USD      ]  Rp       163,126.22  (+1.05%)  -  IDR (kurs: Rp 17,273)


$MATIC-USD: possibly delisted; no price data found  (period=6mo)

1 Failed download:
['MATIC-USD']: possibly delisted; no price data found  (period=6mo)
$MATIC-USD: possibly delisted; no price data found  (period=6mo)

1 Failed download:
['MATIC-USD']: possibly delisted; no price data found  (period=6mo)
$MATIC-USD: possibly delisted; no price data found  (period=6mo)

1 Failed download:
['MATIC-USD']: possibly delisted; no price data found  (period=6mo)


  [SKIP] MATIC-USD: Tidak ada data untuk ticker: MATIC-USD. Pastikan kode benar 
  [CRYPTO] [DOT-USD       ]  Rp        21,798.53  (+1.24%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [SHIB-USD      ]  Rp             0.11  (+0.35%)  -  IDR (kurs: Rp 17,273)


...

  [SKIP] UNI-USD: Tidak ada data untuk ticker: UNI-USD. Pastikan kode benar (c
  [CRYPTO] [ATOM-USD      ]  Rp        34,943.28  (-0.42%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [TRX-USD       ]  Rp         5,590.92  (-0.10%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [NEAR-USD      ]  Rp        24,095.83  (-0.49%)  -  IDR (kurs: Rp 17,273)


$APT-USD: possibly delisted; no price data found  (period=6mo)

1 Failed download:
['APT-USD']: possibly delisted; no price data found  (period=6mo)
$APT-USD: possibly delisted; no price data found  (period=6mo)

1 Failed download:
['APT-USD']: possibly delisted; no price data found  (period=6mo)
$APT-USD: possibly delisted; no price data found  (period=6mo)

1 Failed download:
['APT-USD']: possibly delisted; no price data found  (period=6mo)
$APT-USD: possibly delisted; no price data found  (period=6mo)

1 Failed download:
['APT-USD']: possibly delisted; no price data found  (period=6mo)


  [SKIP] APT-USD: Tidak ada data untuk ticker: APT-USD. Pastikan kode benar (c
  [CRYPTO] [OP-USD        ]  Rp         2,176.40  (+0.75%)  -  IDR (kurs: Rp 17,273)
  [CRYPTO] [ARB-USD       ]  Rp            13.08  (+0.00%)  -  IDR (kurs: Rp 17,273)
  Berhasil: 17/20 aset.  Gagal: ['MATIC-USD', 'UNI-USD', 'APT-USD']


  SCREENER BUDGET -- Modal Tersedia: Rp 1,000,000
  [x] [BTC-USD       ] Rp 1,344,773,491.45 | 1 Unit: Rp 1,344,773,491.45 -> MAHAL
  [x] [ETH-USD       ] Rp    40,251,791.44 | 1 Unit: Rp  40,251,791.44 -> MAHAL
  [x] [BNB-USD       ] Rp    10,905,653.93 | 1 Unit: Rp  10,905,653.93 -> MAHAL
  [x] [SOL-USD       ] Rp     1,486,859.87 | 1 Unit: Rp   1,486,859.87 -> MAHAL
  [v] [XRP-USD       ] Rp        24,617.48 | 1 Unit: Rp      24,617.48 -> LOLOS
  [v] [ADA-USD       ] Rp         4,366.61 | 1 Unit: Rp       4,366.61 -> LOLOS
  [v] [AVAX-USD      ] Rp       163,057.11 | 1 Unit: Rp     163,057.11 -> LOLOS
  [v] [DOGE-USD      ] Rp         1,702.95 | 1 Unit: Rp       1,702.9

,Ticker,Nama,Sektor,Harga (IDR),Min. Beli (IDR),Budget OK?,PER,PBV,ROE (%),Skor (0-100)
1,LINK-USD,Chainlink USD,--,1.631262e+05,1.631262e+05,[v],--,--,--,36.0
2,DOT-USD,Polkadot USD,--,2.179853e+04,2.179853e+04,[v],--,--,--,36.0
3,OP-USD,Optimism USD,--,2.176400e+03,2.176400e+03,[v],--,--,--,36.0
4,NEAR-USD,NEAR Protocol USD,--,2.409583e+04,2.409583e+04,[v],--,--,--,36.0
5,TRX-USD,TRON USD,--,5.590920e+03,5.590920e+03,[v],--,--,--,36.0
6,ATOM-USD,Cosmos USD,--,3.494328e+04,3.494328e+04,[v],--,--,--,36.0
7,LTC-USD,Litecoin USD,--,9.691880e+05,9.691880e+05,[v],--,--,--,36.0
8,SHIB-USD,Shiba Inu USD,--,1.100000e-01,1.100000e-01,[v],--,--,--,36.0
9,ARB-USD,ARbit USD,--,1.308000e+01,1.308000e+01,[v],--,--,--,36.0
10,DOGE-USD,Dogecoin USD,--,1.702950e+03,1.702950e+03,[v],--,--,--,36.0



  Crypto Terjangkau Terbaik:
  -> LINK-USD | Harga: Rp 163,126.22/unit
     Skor: 36.0/100


---
## Cell 8D - SCREENER FOREX & KOMODITAS

> Hanya input budget, scan 12 pasangan forex + emas

In [ ]:
# ======================================================================
# CELL 8D: SCREENER FOREX & KOMODITAS -- Fix v3: tidak lagi error
# ======================================================================

DAFTAR_FOREX = [
    # ── MAJOR PAIRS ─────────────────────────────────────
    "EURUSD=X", "GBPUSD=X", "USDJPY=X", "AUDUSD=X", "USDCHF=X",
    "USDCAD=X", "NZDUSD=X", "EURGBP=X", "EURJPY=X", "GBPJPY=X",
    # ── ASIAN & EM PAIRS ────────────────────────────────
    "USDIDR=X", "USDSGD=X", "USDMYR=X", "USDTHB=X", "USDPHP=X",
    "USDCNY=X", "USDINR=X", "USDKRW=X", "USDVND=X", "USDBRL=X",
    # ── KOMODITAS ───────────────────────────────────────
    "XAUUSD=X",  # Emas
    "XAGUSD=X",  # Perak
    "XPTUSD=X",  # Platinum
    "XPDUSD=X",  # Palladium
    # ── KOMODITAS VIA ETF/FUTURES YFINANCE ──────────────
    "CL=F",      # Crude Oil WTI
    "BZ=F",      # Brent Crude Oil
    "GC=F",      # Gold Futures
    "SI=F",      # Silver Futures
    "HG=F",      # Copper Futures
    "NG=F",      # Natural Gas
    "ZC=F",      # Corn Futures
    "ZW=F",      # Wheat Futures
    "ZS=F",      # Soybean Futures
    "KC=F",      # Coffee Futures
    "CC=F",      # Cocoa Futures
    "SB=F",      # Sugar Futures
]

print("=" * 65)
print("  MAESTRO -- SCREENER FOREX & KOMODITAS")
print("  Scan otomatis 12 pasangan forex + emas + perak")
print("=" * 65)

try:
    budget_str   = input("\n  Budget Anda (Rupiah) [contoh: 2000000] : ").strip()
    budget_forex = float(budget_str.replace(",", "").replace(".", ""))
except ValueError:
    budget_forex = 2_000_000
    print(f"  [!] Menggunakan default: Rp {budget_forex:,.0f}")

print(f"\n  Budget   : Rp {budget_forex:,.0f}")
print(f"  Scanning : {len(DAFTAR_FOREX)} pasangan forex...\n")

data_forex  = tarik_data_batch(DAFTAR_FOREX, period="6mo")
tabel_forex = screener_budget(data_forex, budget=budget_forex)

print("\n" + "=" * 65)
print("  HASIL SCREENER FOREX & KOMODITAS")
print("=" * 65)
display(tabel_forex)

lolos_f = tabel_forex[tabel_forex["Budget OK?"] == "[v]"]
if not lolos_f.empty:
    print(f"\n  Forex dalam Budget Rp {budget_forex:,.0f}:")
    for _, row in lolos_f.head(3).iterrows():
        print(f"  -> {row['Ticker']:<14} | Rp {row['Harga (IDR)']:>16,.2f}/unit")
else:
    print(f"\n  [!] Semua forex melebihi budget Rp {budget_forex:,.0f}")
    print("  Saran: Forex memerlukan modal minimal Rp 2-5 juta per unit (kecuali pair kecil).")

  MAESTRO -- SCREENER FOREX & KOMODITAS
  Scan otomatis 12 pasangan forex + emas + perak

  Budget   : Rp 50,000
  Scanning : 12 pasangan forex...


  MAESTRO DATA ENGINE - MENARIK DATA MULTI-ASET
  [FOREX]  [EURUSD=X      ]  Rp        20,254.46  (+0.17%)  -  IDR (kurs: Rp 17,273)
  [FOREX]  [GBPUSD=X      ]  Rp        23,368.42  (+0.46%)  -  IDR (kurs: Rp 17,273)
  [FOREX]  [USDJPY=X      ]  Rp     2,752,158.78  (-0.10%)  -  IDR (kurs: Rp 17,273)
  [FOREX]  [AUDUSD=X      ]  Rp        12,355.51  (+0.33%)  -  IDR (kurs: Rp 17,273)
  [FOREX]  [USDCHF=X      ]  Rp        13,547.73  (-0.06%)  -  IDR (kurs: Rp 17,273)
  [FOREX]  [USDCAD=X      ]  Rp        23,608.74  (-0.22%)  -  IDR (kurs: Rp 17,273)
  [FOREX]  [NZDUSD=X      ]  Rp        10,157.00  (+0.45%)  -  IDR (kurs: Rp 17,273)


HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: XAUUSD=X"}}}
$XAUUSD=X: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['XAUUSD=X']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
HTTP Error 404: {"quoteSummary":{"result":null,"error":{"code":"Not Found","description":"Quote not found for symbol: XAUUSD=X"}}}
$XAUUSD=X: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['XAUUSD=X']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
$XAUUSD=X: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['XAUUSD=X']: possibly delisted; no price data found  (period=6mo) (Yaho

  [SKIP] XAUUSD=X: Tidak ada data untuk ticker: XAUUSD=X. Pastikan kode benar (
  [FOREX]  [USDIDR=X      ]  Rp   298,356,529.00  (-0.32%)  -  IDR (kurs: Rp 17,273)
  [FOREX]  [USDSGD=X      ]  Rp        22,036.89  (-0.16%)  -  IDR (kurs: Rp 17,273)
  [FOREX]  [USDMYR=X      ]  Rp        68,435.62  (+0.03%)  -  IDR (kurs: Rp 17,273)


$XAGUSD=X: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['XAGUSD=X']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
$XAGUSD=X: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['XAGUSD=X']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
$XAGUSD=X: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['XAGUSD=X']: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")
$XAGUSD=X: possibly delisted; no price data found  (period=6mo) (Yahoo error = "No data found, symbol may be delisted")

1 Failed download:
['XAGUSD=X']: possibly delisted; no price data found  (period=6mo) (Yah

  [SKIP] XAGUSD=X: Tidak ada data untuk ticker: XAGUSD=X. Pastikan kode benar (
  Berhasil: 10/12 aset.  Gagal: ['XAUUSD=X', 'XAGUSD=X']


  SCREENER BUDGET -- Modal Tersedia: Rp 50,000
  [v] [EURUSD=X      ] Rp        20,254.46 | 1 Unit: Rp      20,254.46 -> LOLOS
  [v] [GBPUSD=X      ] Rp        23,368.42 | 1 Unit: Rp      23,368.42 -> LOLOS
  [x] [USDJPY=X      ] Rp     2,752,158.78 | 1 Unit: Rp   2,752,158.78 -> MAHAL
  [v] [AUDUSD=X      ] Rp        12,355.51 | 1 Unit: Rp      12,355.51 -> LOLOS
  [v] [USDCHF=X      ] Rp        13,547.73 | 1 Unit: Rp      13,547.73 -> LOLOS
  [v] [USDCAD=X      ] Rp        23,608.74 | 1 Unit: Rp      23,608.74 -> LOLOS
  [v] [NZDUSD=X      ] Rp        10,157.00 | 1 Unit: Rp      10,157.00 -> LOLOS
  [x] [USDIDR=X      ] Rp   298,356,529.00 | 1 Unit: Rp 298,356,529.00 -> MAHAL
  [v] [USDSGD=X      ] Rp        22,036.89 | 1 Unit: Rp      22,036.89 -> LOLOS
  [x] [USDMYR=X      ] Rp        68,435.62 | 1 Unit: Rp      68,435.62 -> MAHAL

  Screener se

,Ticker,Nama,Sektor,Harga (IDR),Min. Beli (IDR),Budget OK?,PER,PBV,ROE (%),Skor (0-100)
1,EURUSD=X,EUR/USD,--,2.025446e+04,2.025446e+04,[v],--,--,--,36.0
2,GBPUSD=X,GBP/USD,--,2.336842e+04,2.336842e+04,[v],--,--,--,36.0
3,AUDUSD=X,AUD/USD,--,1.235551e+04,1.235551e+04,[v],--,--,--,36.0
4,USDCHF=X,USD/CHF,--,1.354773e+04,1.354773e+04,[v],--,--,--,36.0
5,USDCAD=X,USD/CAD,--,2.360874e+04,2.360874e+04,[v],--,--,--,36.0
6,NZDUSD=X,NZD/USD,--,1.015700e+04,1.015700e+04,[v],--,--,--,36.0
7,USDSGD=X,USD/SGD,--,2.203689e+04,2.203689e+04,[v],--,--,--,36.0
8,USDJPY=X,USD/JPY,--,2.752159e+06,2.752159e+06,[x],--,--,--,0.0
9,USDIDR=X,USD/IDR,--,2.983565e+08,2.983565e+08,[x],--,--,--,0.0
10,USDMYR=X,USD/MYR,--,6.843562e+04,6.843562e+04,[x],--,--,--,0.0



  Forex dalam Budget Rp 50,000:
  -> EURUSD=X       | Rp        20,254.46/unit
  -> GBPUSD=X       | Rp        23,368.42/unit
  -> AUDUSD=X       | Rp        12,355.51/unit


---
## Cell 9 - BUILD APLIKASI ANDROID (APK)

**Cara kerja:**
1. Cell ini menghasilkan file `Maestro_App.py` (Streamlit) + `requirements.txt`
2. Untuk **deploy ke web** (gratis): Upload ke Streamlit Cloud / Hugging Face Spaces
3. Untuk **APK Android**: Gunakan tool WebView APK builder (gratis) yang wrap URL Streamlit
4. Semua fitur analisis lengkap tersedia di app

> **Panduan lengkap tersedia di output cell ini setelah dijalankan.**

In [21]:
# ======================================================================
# CELL 9: BUILD APLIKASI MAESTRO PRO MAX — UI PREMIUM FULL FITUR
# Ganti seluruh Cell 9 lama dengan cell ini
# ======================================================================

import os

print("  Membangun Maestro Pro Max...")

APP_CODE = r'''
import streamlit as st
import yfinance as yf
import pandas as pd
import numpy as np
import requests, random, time, re, os
from scipy.signal import find_peaks
from urllib.parse import quote
from bs4 import BeautifulSoup
import plotly.graph_objects as go
from plotly.subplots import make_subplots

# ══════════════════════════════════════════════════════════════════════
# CONFIG & TEMA
# ══════════════════════════════════════════════════════════════════════
st.set_page_config(
    page_title="Maestro Trading Intelligence",
    page_icon="▲",
    layout="wide",
    initial_sidebar_state="expanded"
)

st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Syne:wght@400;600;700;800&family=JetBrains+Mono:wght@300;400;600&display=swap');

:root {
    --bg:       #080c12;
    --bg2:      #0d1420;
    --bg3:      #111a2b;
    --border:   rgba(0,180,255,0.12);
    --accent:   #00c8ff;
    --green:    #00ff9d;
    --red:      #ff3d6e;
    --gold:     #ffd166;
    --text:     #e8edf5;
    --muted:    #5a6a82;
    --card-bg:  rgba(13,20,32,0.85);
}

* { box-sizing: border-box; }
html, body, [class*="css"] {
    font-family: 'JetBrains Mono', monospace;
    background: var(--bg) !important;
    color: var(--text);
}

/* ── SCROLLBAR ── */
::-webkit-scrollbar { width: 4px; }
::-webkit-scrollbar-track { background: var(--bg); }
::-webkit-scrollbar-thumb { background: var(--accent); border-radius: 2px; }

/* ── SIDEBAR ── */
section[data-testid="stSidebar"] {
    background: var(--bg2) !important;
    border-right: 1px solid var(--border);
}
section[data-testid="stSidebar"] * { font-family: 'JetBrains Mono', monospace !important; }

/* ── MAIN HEADER ── */
.maestro-header {
    display: flex; align-items: center; gap: 16px;
    padding: 28px 0 20px;
    border-bottom: 1px solid var(--border);
    margin-bottom: 24px;
}
.maestro-logo {
    width: 44px; height: 44px;
    background: linear-gradient(135deg, var(--accent), var(--green));
    border-radius: 10px;
    display: flex; align-items: center; justify-content: center;
    font-family: 'Syne', sans-serif;
    font-size: 1.3rem; font-weight: 800; color: #080c12;
}
.maestro-title {
    font-family: 'Syne', sans-serif;
    font-size: 1.6rem; font-weight: 800;
    background: linear-gradient(90deg, var(--accent), var(--green));
    -webkit-background-clip: text; -webkit-text-fill-color: transparent;
    line-height: 1;
}
.maestro-sub { font-size: 0.72rem; color: var(--muted); margin-top: 3px; letter-spacing: 0.08em; }

/* ── METRIC CARDS ── */
.metric-grid { display: grid; grid-template-columns: repeat(5, 1fr); gap: 12px; margin-bottom: 20px; }
.metric-card {
    background: var(--card-bg);
    border: 1px solid var(--border);
    border-radius: 14px;
    padding: 16px 18px;
    position: relative; overflow: hidden;
}
.metric-card::before {
    content: ''; position: absolute; top: 0; left: 0; right: 0; height: 2px;
    background: linear-gradient(90deg, var(--accent), transparent);
}
.metric-label { font-size: 0.65rem; color: var(--muted); letter-spacing: 0.1em; text-transform: uppercase; margin-bottom: 6px; }
.metric-value { font-family: 'Syne', sans-serif; font-size: 1.4rem; font-weight: 700; line-height: 1; }
.metric-delta { font-size: 0.72rem; margin-top: 4px; }
.metric-up   { color: var(--green); }
.metric-dn   { color: var(--red); }
.metric-neu  { color: var(--gold); }

/* ── SECTION HEADER ── */
.sec-header {
    font-family: 'Syne', sans-serif;
    font-size: 0.75rem; font-weight: 700;
    letter-spacing: 0.15em; text-transform: uppercase;
    color: var(--accent); margin: 28px 0 12px;
    display: flex; align-items: center; gap: 8px;
}
.sec-header::after {
    content: ''; flex: 1; height: 1px;
    background: var(--border);
}

/* ── PANEL ── */
.panel {
    background: var(--card-bg);
    border: 1px solid var(--border);
    border-radius: 14px;
    padding: 20px;
}

/* ── TRADING PLAN CARDS ── */
.tp-card {
    border-radius: 10px; padding: 12px 16px;
    margin-bottom: 8px; font-size: 0.82rem;
    display: flex; justify-content: space-between; align-items: center;
}
.tp-buy  { background: rgba(0,255,157,0.07); border: 1px solid rgba(0,255,157,0.25); }
.tp-sell { background: rgba(255,61,110,0.07); border: 1px solid rgba(255,61,110,0.25); }
.tp-wait { background: rgba(255,209,102,0.05); border: 1px solid rgba(255,209,102,0.15); }
.tp-label { font-size: 0.65rem; letter-spacing: 0.1em; text-transform: uppercase; }
.tp-label-buy  { color: var(--green); }
.tp-label-sell { color: var(--red); }
.tp-label-wait { color: var(--gold); }
.tp-val { font-family: 'Syne', sans-serif; font-weight: 700; font-size: 1.0rem; }

/* ── SCHEDULE ROWS ── */
.sched-row {
    display: flex; align-items: center; gap: 12px;
    padding: 9px 12px; border-radius: 8px;
    margin-bottom: 6px; font-size: 0.78rem;
    border: 1px solid transparent;
}
.sched-day  { color: var(--muted); width: 90px; flex-shrink: 0; font-size: 0.7rem; }
.sched-buy  { background: rgba(0,255,157,0.06); border-color: rgba(0,255,157,0.2); }
.sched-sell { background: rgba(255,61,110,0.06); border-color: rgba(255,61,110,0.2); }
.sched-wait { background: rgba(255,255,255,0.02); border-color: transparent; }
.badge {
    font-size: 0.6rem; letter-spacing: 0.08em; padding: 2px 8px;
    border-radius: 20px; font-weight: 700; flex-shrink: 0;
}
.badge-buy  { background: rgba(0,255,157,0.15); color: var(--green); }
.badge-sell { background: rgba(255,61,110,0.15); color: var(--red); }
.badge-wait { background: rgba(255,255,255,0.06); color: var(--muted); }

/* ── FUNDAMENTAL GRID ── */
.fund-grid { display: grid; grid-template-columns: 1fr 1fr; gap: 8px; }
.fund-row {
    display: flex; justify-content: space-between;
    padding: 8px 12px; background: rgba(255,255,255,0.03);
    border-radius: 8px; font-size: 0.78rem; border: 1px solid var(--border);
}
.fund-key { color: var(--muted); }
.fund-val { font-weight: 600; }
.fund-ok  { color: var(--green); }
.fund-mid { color: var(--gold); }
.fund-bad { color: var(--red); }

/* ── SKOR RING ── */
.skor-wrap { display: flex; align-items: center; gap: 16px; padding: 16px 0; }
.skor-ring {
    width: 80px; height: 80px; border-radius: 50%;
    display: flex; align-items: center; justify-content: center;
    font-family: 'Syne', sans-serif; font-size: 1.4rem; font-weight: 800;
    flex-shrink: 0;
}
.skor-ok  { background: conic-gradient(var(--green) 0%, rgba(0,255,157,0.1) 0%); border: 3px solid var(--green); color: var(--green); }
.skor-mid { border: 3px solid var(--gold); color: var(--gold); }
.skor-bad { border: 3px solid var(--red);  color: var(--red); }

/* ── NEWS ── */
.news-item {
    padding: 10px 14px; border-radius: 8px; margin-bottom: 6px;
    font-size: 0.78rem; border-left: 3px solid;
}
.news-pos { border-color: var(--green); background: rgba(0,255,157,0.04); }
.news-neg { border-color: var(--red);   background: rgba(255,61,110,0.04); }
.news-neu { border-color: var(--muted); background: rgba(255,255,255,0.02); }
.news-badge { font-size: 0.6rem; font-weight: 700; letter-spacing: 0.08em; }
.news-src  { font-size: 0.65rem; color: var(--muted); margin-top: 3px; }

/* ── PLAN TABLE ── */
.plan-row {
    display: grid; grid-template-columns: 100px 80px 1fr 120px;
    gap: 12px; padding: 10px 14px; border-radius: 8px; margin-bottom: 4px;
    font-size: 0.78rem; align-items: center;
}
.plan-row-buy  { background: rgba(0,255,157,0.05); border: 1px solid rgba(0,255,157,0.15); }
.plan-row-sell { background: rgba(255,61,110,0.05); border: 1px solid rgba(255,61,110,0.15); }
.plan-header {
    display: grid; grid-template-columns: 100px 80px 1fr 120px;
    gap: 12px; padding: 6px 14px; margin-bottom: 8px;
    font-size: 0.62rem; color: var(--muted); letter-spacing: 0.1em; text-transform: uppercase;
}

/* ── BANDARMOLOGI ── */
.bandar-card {
    border-radius: 10px; padding: 14px 18px;
    display: flex; align-items: center; gap: 14px;
}
.bandar-akum { background: rgba(0,255,157,0.08); border: 1px solid rgba(0,255,157,0.3); }
.bandar-dist { background: rgba(255,61,110,0.08); border: 1px solid rgba(255,61,110,0.3); }
.bandar-neu  { background: rgba(255,255,255,0.03); border: 1px solid var(--border); }
.bandar-dot  { width: 14px; height: 14px; border-radius: 50%; flex-shrink: 0; }

/* ── SCREENER TABLE ── */
[data-testid="stDataFrame"] { border-radius: 12px !important; overflow: hidden; }

/* ── SIDEBAR ITEMS ── */
.stRadio label { font-size: 0.8rem !important; }
.stTextInput input, .stSelectbox select {
    background: var(--bg3) !important;
    border: 1px solid var(--border) !important;
    border-radius: 8px !important;
    color: var(--text) !important;
    font-family: 'JetBrains Mono', monospace !important;
}
.stButton button {
    background: linear-gradient(135deg, var(--accent), var(--green)) !important;
    color: #080c12 !important; font-weight: 700 !important;
    border: none !important; border-radius: 10px !important;
    font-family: 'Syne', sans-serif !important;
    letter-spacing: 0.05em !important;
}
.stButton button:hover { opacity: 0.88 !important; }

/* ── HIDE STREAMLIT CHROME ── */
#MainMenu, footer, header { visibility: hidden; }
</style>
""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════════════════════════
# KONSTANTA
# ══════════════════════════════════════════════════════════════════════
BROWSER_HEADERS = [
    {"User-Agent": "Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 Chrome/122.0.0.0 Safari/537.36",
     "Accept-Language": "id-ID,id;q=0.9,en-US;q=0.8"},
    {"User-Agent": "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/605.1.15 Version/17.3 Safari/605.1.15",
     "Accept-Language": "id-ID,id;q=0.9"},
]
KATA_POSITIF = [
    # Bahasa Indonesia — Pergerakan & Kinerja
    "naik","untung","laba","profit","surplus","tumbuh","meningkat","menguat","melesat",
    "melompat","meroket","rebound","recover","bangkit","pulih","ekspansi","berkembang",
    "solid","kuat","optimis","prospek cerah","positif","harapan","peluang","potensi",
    "efisien","produktif","inovatif","bertumbuh","perbaikan","pemulihan","apresiasi",
    # Aksi Korporasi
    "dividen","buyback","akuisisi","merger","spin-off","rights issue","stock split",
    "ekspansi bisnis","kontrak baru","proyek baru","kemitraan","kolaborasi","investasi",
    # Pasar & Teknikal
    "bullish","breakout","all time high","ath","golden cross","uptrend","rally",
    "volume tinggi","akumulasi","demand tinggi","buy signal","oversold","support kuat",
    # Bahasa Inggris — Umum
    "gain","rise","up","beat","record","growth","outperform","upgrade","strong",
    "exceed","positive","boom","surge","jump","climb","advance","improve","recovery",
    "momentum","robust","resilient","optimistic","buy","accumulate","recommend",
]

KATA_NEGATIF = [
    # Bahasa Indonesia — Pergerakan & Kinerja
    "turun","rugi","loss","melemah","merosot","anjlok","tertekan","jatuh","ambles",
    "terpuruk","stagnan","koreksi","penurunan","kemerosotan","gagal","kerugian",
    "defisit","minus","negatif","pesimis","khawatir","risiko","ancaman","tekanan",
    "lesap","terkikis","menyusut","memburuk","tergelincir","tumbang","kolaps",
    # Aksi Korporasi & Hukum
    "bangkrut","pailit","delisting","suspensi","pembekuan","likuidasi","restrukturisasi",
    "gugatan","sanksi","denda","korupsi","skandal","fraud","manipulasi","penggelapan",
    "suap","investigasi","penyidikan","pemeriksaan","pelanggaran","sengketa","tuntutan",
    # Makro & Industri
    "resesi","inflasi tinggi","stagflasi","krisis","gejolak","ketidakpastian","volatile",
    "perang","konflik","lockdown","pandemi","bencana","banjir","gempa","gangguan",
    # Pasar & Teknikal
    "bearish","downtrend","death cross","breakdown","resistance kuat","overbought",
    "sell signal","distribusi","supply tinggi","panic selling","margin call","short",
    # Bahasa Inggris — Umum
    "drop","fall","down","miss","negative","weak","crash","plunge","decline","tumble",
    "underperform","downgrade","sell","loss","deficit","bankruptcy","default","risk",
    "warning","concern","pressure","volatile","uncertainty","layoff","cut","revision",
]

HARI_ID = {"Monday":"Senin","Tuesday":"Selasa","Wednesday":"Rabu",
           "Thursday":"Kamis","Friday":"Jumat","Saturday":"Sabtu","Sunday":"Minggu"}

# ══════════════════════════════════════════════════════════════════════
# FUNGSI DATA
# ══════════════════════════════════════════════════════════════════════
@st.cache_data(ttl=600)
def tarik_data(ticker, period="1y"):
    ticker = ticker.strip().upper()
    is_jk  = ticker.endswith(".JK")
    df = None
    for _ in range(3):
        try:
            df = yf.download(ticker, period=period, interval="1d",
                             progress=False, auto_adjust=False)
            if not df.empty: break
            time.sleep(random.uniform(0.3, 0.7))
        except Exception:
            time.sleep(0.5)
    if df is None or df.empty:
        return None, None, 1.0
    if isinstance(df.columns, pd.MultiIndex):
        df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    if "Datetime" in df.columns: df.rename(columns={"Datetime": "Date"}, inplace=True)
    df["Date"] = pd.to_datetime(df["Date"]).dt.normalize()
    kurs = 1.0
    if not is_jk:
        try:
            kr = yf.download("IDR=X", period="5d", progress=False, auto_adjust=True)
            if isinstance(kr.columns, pd.MultiIndex): kr.columns = kr.columns.get_level_values(0)
            kurs = float(kr["Close"].dropna().iloc[-1])
            if kurs < 1000: kurs = 16200.0
        except Exception: kurs = 16200.0
        for c in ["Open","High","Low","Close"]: df[c] = pd.to_numeric(df[c], errors="coerce") * kurs
    else:
        for c in ["Open","High","Low","Close"]: df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ["Open","High","Low","Close"]: df[f"{c}_IDR"] = df[c]
    if "Volume" not in df.columns: df["Volume"] = 0.0
    df["Volume"] = pd.to_numeric(df["Volume"], errors="coerce").fillna(0)
    df = df.dropna(subset=["Close_IDR"]).sort_values("Date").reset_index(drop=True)
    win = min(20, max(5, len(df)//3))
    df["Support"]    = df["Low_IDR"].rolling(win).min().shift(1)
    df["Resistance"] = df["High_IDR"].rolling(win).max().shift(1)
    vm = df["Volume"].rolling(20).mean()
    df["Vol_Spike"] = df["Volume"] > (vm * 1.5)
    df["Is_Up"]     = df["Close_IDR"] > df["Open_IDR"]
    df["Bandar"]    = df.apply(lambda r:
        "AKUMULASI" if r["Vol_Spike"] and r["Is_Up"] else
        ("DISTRIBUSI" if r["Vol_Spike"] and not r["Is_Up"] else "NETRAL"), axis=1)
    # ATR
    h = df["High_IDR"]; l = df["Low_IDR"]; c = df["Close_IDR"]; cp = c.shift(1)
    tr = pd.concat([h-l, (h-cp).abs(), (l-cp).abs()], axis=1).max(axis=1)
    df["ATR"] = tr.rolling(14).mean()
    return df, is_jk, kurs


@st.cache_data(ttl=3600)
def hitung_siklus(df):
    close = df["Close_IDR"].values
    dates = pd.to_datetime(df["Date"].values)
    pr    = close.max() - close.min()
    if pr == 0: pr = close.mean() * 0.1
    prom  = pr * 0.02
    peaks,   _ = find_peaks(close,  prominence=prom, distance=5)
    valleys, _ = find_peaks(-close, prominence=prom, distance=5)
    def get_cyc(idxs):
        if len(idxs) < 2: return None, None
        gaps = [(dates[idxs[i+1]] - dates[idxs[i]]).days for i in range(len(idxs)-1)]
        avg  = int(np.mean(gaps))
        ds   = (dates[-1] - dates[idxs[-1]]).days
        dtn  = avg - (ds % avg)
        nxt  = dates[-1] + pd.Timedelta(days=int(dtn))
        return avg, nxt
    ap, np_ = get_cyc(peaks)
    av, nv  = get_cyc(valleys)
    return ap, np_, av, nv, peaks, valleys


@st.cache_data(ttl=3600)
def ambil_fundamental(ticker):
    try:
        info = yf.Ticker(ticker).info
        return {
            "PER":    info.get("trailingPE"),
            "PBV":    info.get("priceToBook"),
            "ROE":    info.get("returnOnEquity"),
            "Margin": info.get("profitMargins"),
            "DE":     info.get("debtToEquity"),
            "Beta":   info.get("beta"),
            "Div":    info.get("dividendYield"),
            "EPS":    info.get("trailingEps"),
            "52H":    info.get("fiftyTwoWeekHigh"),
            "52L":    info.get("fiftyTwoWeekLow"),
            "MCap":   info.get("marketCap"),
            "Nama":   info.get("longName", ticker),
            "Sektor": info.get("sector", "--"),
            "Desk":   (info.get("longBusinessSummary","") or "")[:300],
        }
    except Exception: return {}


def hitung_skor(fund):
    s = 0
    def to_float(v):
        try: return float(v)
        except (TypeError, ValueError): return None
    per = to_float(fund.get("PER"))
    if per and per > 0:
        s += 25 if per<10 else (20 if per<15 else (14 if per<20 else (8 if per<25 else 2)))
    else: s += 10
    pbv = to_float(fund.get("PBV"))
    if pbv and pbv > 0:
        s += 20 if pbv<1 else (15 if pbv<2 else (10 if pbv<3 else 4))
    else: s += 8
    roe = to_float(fund.get("ROE"))
    if roe: s += 25 if roe*100>=25 else (20 if roe*100>=15 else (12 if roe*100>=10 else 5))
    else: s += 8
    m = to_float(fund.get("Margin"))
    if m: s += 15 if m*100>=20 else (10 if m*100>=10 else (5 if m*100>=5 else 0))
    else: s += 5
    de = to_float(fund.get("DE"))
    if de is not None: s += 15 if de<30 else (10 if de<60 else (5 if de<100 else 0))
    else: s += 5
    return min(s, 100)


def fmt_mcap(v):
    if not v: return "--"
    if v >= 1e12: return f"Rp {v/1e12:.1f}T"
    if v >= 1e9:  return f"Rp {v/1e9:.1f}M"
    return f"Rp {v/1e6:.0f}jt"


@st.cache_data(ttl=900)
def ambil_berita(ticker):
    semua = []
    query = ticker.replace(".JK","").replace("-USD","").replace("=X","")
    try:
        q   = quote(f"{query} saham investasi")
        url = f"https://news.google.com/rss/search?q={q}&hl=id&gl=ID&ceid=ID:id"
        r   = requests.get(url, headers=random.choice(BROWSER_HEADERS), timeout=8)
        if r.status_code == 200:
            soup = BeautifulSoup(r.text, "lxml-xml")
            for item in soup.find_all("item")[:8]:
                t  = item.find("title")
                d  = item.find("pubDate")
                sr = item.find("source")
                if t: semua.append({
                    "judul":  t.text.strip()[:130],
                    "tgl":    d.text[:10] if d else "",
                    "sumber": sr.text.strip() if sr else "Google News"
                })
    except Exception: pass
    if len(semua) < 3:
        try:
            for item in (yf.Ticker(ticker).news or [])[:6]:
                c = item.get("content", {})
                t = c.get("title","") if isinstance(c, dict) else item.get("title","")
                p = c.get("provider",{}).get("displayName","Yahoo") if isinstance(c, dict) else item.get("publisher","Yahoo")
                if t: semua.append({"judul": t[:130], "tgl": "", "sumber": p})
        except Exception: pass
    return semua


def sentimen_teks(judul):
    j = judul.lower()
    p = sum(1 for k in KATA_POSITIF if k in j)
    n = sum(1 for k in KATA_NEGATIF if k in j)
    if p > n: return "pos"
    if n > p: return "neg"
    return "neu"


def buat_jadwal_30hari(np_, nv, ap, av):
    jadwal = []
    today  = pd.Timestamp.now().normalize()
    end    = today + pd.Timedelta(days=30)
    if np_ and ap:
        t = pd.Timestamp(np_)
        while t <= end:
            if t > today: jadwal.append({"tgl": t, "type": "peak"})
            t += pd.Timedelta(days=ap)
    if nv and av:
        t = pd.Timestamp(nv)
        while t <= end:
            if t > today: jadwal.append({"tgl": t, "type": "valley"})
            t += pd.Timedelta(days=av)
    return sorted(jadwal, key=lambda x: x["tgl"])


def hitung_sr_kuat(df, lookback=20):
    low  = df["Low_IDR"].values
    high = df["High_IDR"].values
    sups, ress = [], []
    for i in range(lookback, len(df)):
        sups.append(float(low[i-lookback:i].min()))
        ress.append(float(high[i-lookback:i].max()))
    def cluster(pts, n=3):
        if not pts: return []
        pts = sorted(set(round(p, -2) for p in pts))
        cls, used = [], set()
        for p in pts:
            if p in used: continue
            grp = [x for x in pts if abs(x-p)/max(p,1) < 0.02]
            cls.append(float(sum(grp)/len(grp)))
            for x in grp: used.add(x)
        return sorted(cls)[-n:]
    return cluster(sups), cluster(ress)

# ══════════════════════════════════════════════════════════════════════
# SIDEBAR NAVIGASI
# ══════════════════════════════════════════════════════════════════════
with st.sidebar:
    st.markdown("""
    <div style='padding:16px 0 20px;'>
      <div style='font-family:Syne,sans-serif;font-size:1.1rem;font-weight:800;
                  background:linear-gradient(90deg,#00c8ff,#00ff9d);
                  -webkit-background-clip:text;-webkit-text-fill-color:transparent;'>
        ▲ MAESTRO
      </div>
      <div style='font-size:0.65rem;color:#5a6a82;letter-spacing:0.1em;margin-top:2px;'>
        TRADING INTELLIGENCE v3.0
      </div>
    </div>
    """, unsafe_allow_html=True)

    st.markdown("<div style='font-size:0.65rem;color:#5a6a82;letter-spacing:0.1em;margin-bottom:6px;'>MODE OPERASI</div>", unsafe_allow_html=True)
    mode = st.radio("", [
        "▲  Analisis 1 Aset",
        "◈  Screener Saham IDX",
        "◎  Screener Crypto",
        "◇  Screener Forex",
    ], label_visibility="collapsed")

    st.divider()

    if "Analisis" in mode:
        st.markdown("<div style='font-size:0.65rem;color:#5a6a82;letter-spacing:0.1em;margin-bottom:6px;'>KODE ASET</div>", unsafe_allow_html=True)
        ticker  = st.text_input("", "BBCA.JK", label_visibility="collapsed",
                                placeholder="cth: BBCA.JK, BTC-USD, EURUSD=X").upper().strip()

        st.markdown("<div style='font-size:0.65rem;color:#5a6a82;letter-spacing:0.1em;margin:10px 0 6px;'>PERIODE DATA</div>", unsafe_allow_html=True)
        # ── UBAH PERIODE DI SINI ─────────────────────────────────────────
        # Tambah / hapus opsi sesuai kebutuhan: "1mo","2mo","3mo","6mo","1y","2y","5y"
        # index=3 artinya default = "1y" (hitungan dari 0)
        periode = st.selectbox("", ["1mo","2mo","3mo","6mo","1y","2y","5y"],
                               index=0, label_visibility="collapsed")
        # ─────────────────────────────────────────────────────────────────

        run = st.button("ANALISIS SEKARANG", use_container_width=True)

    elif "Saham" in mode:
        st.markdown("<div style='font-size:0.65rem;color:#5a6a82;letter-spacing:0.1em;margin-bottom:6px;'>BUDGET (Rp)</div>", unsafe_allow_html=True)
        budget = st.number_input("", 10000, 100_000_000, 500_000, 50_000, label_visibility="collapsed")
        run    = st.button("SCAN SAHAM IDX", use_container_width=True)

    elif "Crypto" in mode:
        st.markdown("<div style='font-size:0.65rem;color:#5a6a82;letter-spacing:0.1em;margin-bottom:6px;'>BUDGET (Rp)</div>", unsafe_allow_html=True)
        budget = st.number_input("", 10000, 100_000_000, 1_000_000, 100_000, label_visibility="collapsed")
        run    = st.button("SCAN CRYPTO", use_container_width=True)

    elif "Forex" in mode:
        st.markdown("<div style='font-size:0.65rem;color:#5a6a82;letter-spacing:0.1em;margin-bottom:6px;'>BUDGET (Rp)</div>", unsafe_allow_html=True)
        budget = st.number_input("", 10000, 100_000_000, 2_000_000, 500_000, label_visibility="collapsed")
        run    = st.button("SCAN FOREX", use_container_width=True)

    st.divider()
    st.markdown("<div style='font-size:0.62rem;color:#5a6a82;'>Data: yfinance + Google News<br>Refresh: setiap 10 menit</div>", unsafe_allow_html=True)

# ══════════════════════════════════════════════════════════════════════
# HEADER UTAMA
# ══════════════════════════════════════════════════════════════════════
st.markdown("""
<div class="maestro-header">
  <div class="maestro-logo">M</div>
  <div>
    <div class="maestro-title">MAESTRO Trading Intelligence</div>
    <div class="maestro-sub">QUANTITATIVE SYSTEM · TIME-CYCLE · BANDARMOLOGI · BERITA INDONESIA</div>
  </div>
</div>
""", unsafe_allow_html=True)

# ══════════════════════════════════════════════════════════════════════
# MODE: ANALISIS 1 ASET
# ══════════════════════════════════════════════════════════════════════
if "Analisis" in mode:
    if not run:
        st.markdown("""
        <div style='text-align:center;padding:80px 20px;color:#5a6a82;'>
          <div style='font-family:Syne,sans-serif;font-size:2rem;font-weight:800;margin-bottom:12px;'>
            Masukkan kode aset & klik Analisis
          </div>
          <div style='font-size:0.85rem;'>
            Saham IDX: BBCA.JK · TLKM.JK &nbsp;|&nbsp;
            Crypto: BTC-USD · ETH-USD &nbsp;|&nbsp;
            Forex: EURUSD=X · XAUUSD=X &nbsp;|&nbsp;
            US: AAPL · NVDA
          </div>
        </div>
        """, unsafe_allow_html=True)
    else:
        with st.spinner(f"Menganalisis {ticker}..."):
            df, is_jk, kurs = tarik_data(ticker, periode)

        if df is None or df.empty:
            st.error(f"Gagal mengambil data untuk **{ticker}**. Periksa kode ticker dan koneksi internet.")
        else:
            harga  = float(df["Close_IDR"].iloc[-1])
            harga0 = float(df["Close_IDR"].iloc[-2]) if len(df) > 1 else harga
            pct    = (harga - harga0) / max(harga0, 1) * 100
            sup    = float(df["Support"].dropna().iloc[-1])    if not df["Support"].dropna().empty    else harga*0.95
            res    = float(df["Resistance"].dropna().iloc[-1]) if not df["Resistance"].dropna().empty else harga*1.05
            atr_v  = float(df["ATR"].dropna().iloc[-1])        if not df["ATR"].dropna().empty        else (res-sup)*0.3
            bandar = df["Bandar"].iloc[-1]
            vol_l  = float(df["Volume"].iloc[-1])
            vol_m  = float(df["Volume"].rolling(20).mean().iloc[-1]) if len(df) >= 20 else vol_l

            fund   = ambil_fundamental(ticker)
            skor   = hitung_skor(fund)
            ap, np_, av, nv, peaks, valleys = hitung_siklus(df)
            sup_levels, res_levels = hitung_sr_kuat(df)

            # Risk/Reward
            rr_num = (res*0.98 - sup) / max(sup - sup*0.96, 1)

            # ── METRIC CARDS ──────────────────────────────────────────
            pct_cls = "metric-up" if pct >= 0 else "metric-dn"
            pct_sym = "▲" if pct >= 0 else "▼"
            ban_cls = "metric-up" if bandar=="AKUMULASI" else ("metric-dn" if bandar=="DISTRIBUSI" else "metric-neu")
            vol_ratio = vol_l / max(vol_m, 1)

            lot_str = f"Rp {harga*100:,.0f}" if is_jk else f"Rp {harga:,.2f}"
            lot_lbl = "1 LOT (100 LBR)" if is_jk else "1 UNIT"

            st.markdown(f"""
            <div class="metric-grid">
              <div class="metric-card">
                <div class="metric-label">Harga Terakhir</div>
                <div class="metric-value">Rp {harga:,.0f}</div>
                <div class="metric-delta {pct_cls}">{pct_sym} {abs(pct):.2f}% hari ini</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">Skor Fundamental</div>
                <div class="metric-value" style="color:{'#00ff9d' if skor>=70 else ('#ffd166' if skor>=45 else '#ff3d6e')}">{skor}<span style="font-size:0.8rem;color:#5a6a82">/100</span></div>
                <div class="metric-delta metric-neu">{'EXCELLENT' if skor>=75 else ('GOOD' if skor>=55 else ('FAIR' if skor>=35 else 'WEAK'))}</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">Bandarmologi</div>
                <div class="metric-value {ban_cls}" style="font-size:1rem;">{bandar}</div>
                <div class="metric-delta metric-neu">Vol {vol_ratio:.1f}x avg</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">ATR (14)</div>
                <div class="metric-value" style="font-size:1.1rem;">Rp {atr_v:,.0f}</div>
                <div class="metric-delta metric-neu">Volatilitas harian</div>
              </div>
              <div class="metric-card">
                <div class="metric-label">{lot_lbl}</div>
                <div class="metric-value" style="font-size:1.0rem;">{lot_str}</div>
                <div class="metric-delta metric-neu">Kurs Rp {kurs:,.0f}/USD</div>
              </div>
            </div>
            """, unsafe_allow_html=True)

            # ── CHART INTERAKTIF ──────────────────────────────────────
            st.markdown('<div class="sec-header">Chart Interaktif</div>', unsafe_allow_html=True)

            fig = make_subplots(rows=2, cols=1, row_heights=[0.78, 0.22],
                                shared_xaxes=True, vertical_spacing=0.03)

            # Zona beli & jual
            fig.add_trace(go.Scatter(
                x=list(df["Date"])+list(df["Date"][::-1]),
                y=list(df["Support"])+list((df["Support"]*0.97)[::-1]),
                fill="toself", fillcolor="rgba(0,255,157,0.07)",
                line=dict(color="rgba(0,0,0,0)"), name="Zona Beli"), row=1, col=1)
            fig.add_trace(go.Scatter(
                x=list(df["Date"])+list(df["Date"][::-1]),
                y=list(df["Resistance"]*1.03)+list(df["Resistance"][::-1]),
                fill="toself", fillcolor="rgba(255,61,110,0.07)",
                line=dict(color="rgba(0,0,0,0)"), name="Zona Jual"), row=1, col=1)

            # Harga utama
            fig.add_trace(go.Scatter(
                x=df["Date"], y=df["Close_IDR"], mode="lines",
                line=dict(color="#00c8ff", width=2.2),
                name=ticker,
                hovertemplate="<b>%{x|%d %b %Y}</b><br>Rp %{y:,.0f}<extra></extra>"), row=1, col=1)

            # Support & Resistance dinamis
            fig.add_trace(go.Scatter(x=df["Date"], y=df["Support"], mode="lines",
                line=dict(color="#00ff9d", width=1, dash="dot"), name="Support"), row=1, col=1)
            fig.add_trace(go.Scatter(x=df["Date"], y=df["Resistance"], mode="lines",
                line=dict(color="#ff3d6e", width=1, dash="dot"), name="Resistance"), row=1, col=1)

            # Garis S/R kuat horizontal
            x0, x1 = df["Date"].iloc[0], df["Date"].iloc[-1]
            for i, sv in enumerate(sup_levels):
                fig.add_shape(type="line", x0=x0, x1=x1, y0=sv, y1=sv,
                              xref="x", yref="y", line=dict(color="#00ff9d", dash="dashdot", width=1.5))
                fig.add_annotation(x=x1, y=sv, xref="x", yref="y",
                                   text=f"  S{i+1} Rp {sv:,.0f}", showarrow=False,
                                   font=dict(color="#00ff9d", size=9), xanchor="left")
            for i, rv in enumerate(res_levels):
                fig.add_shape(type="line", x0=x0, x1=x1, y0=rv, y1=rv,
                              xref="x", yref="y", line=dict(color="#ff3d6e", dash="dashdot", width=1.5))
                fig.add_annotation(x=x1, y=rv, xref="x", yref="y",
                                   text=f"  R{i+1} Rp {rv:,.0f}", showarrow=False,
                                   font=dict(color="#ff3d6e", size=9), xanchor="left")

            # Puncak & Lembah historis
            if len(peaks):
                fig.add_trace(go.Scatter(
                    x=df["Date"].iloc[peaks], y=df["Close_IDR"].iloc[peaks], mode="markers",
                    marker=dict(symbol="triangle-down", color="#ff3d6e", size=11,
                                line=dict(color="white", width=1)), name="Puncak"), row=1, col=1)
            if len(valleys):
                fig.add_trace(go.Scatter(
                    x=df["Date"].iloc[valleys], y=df["Close_IDR"].iloc[valleys], mode="markers",
                    marker=dict(symbol="triangle-up", color="#00ff9d", size=11,
                                line=dict(color="white", width=1)), name="Lembah"), row=1, col=1)

            # Garis prediksi siklus
            def vline(fig, xdate, color, label):
                xs = pd.Timestamp(xdate).strftime("%Y-%m-%d")
                fig.add_shape(type="line", x0=xs, x1=xs, y0=0, y1=1,
                              yref="paper", xref="x", line=dict(color=color, dash="dash", width=1.2))
                fig.add_annotation(x=xs, y=1.02, yref="paper", xref="x",
                                   text=label, showarrow=False,
                                   font=dict(color=color, size=9), xanchor="center")
            if np_: vline(fig, np_, "#ff3d6e", f"Puncak ~{pd.Timestamp(np_).strftime('%d %b')}")
            if nv:  vline(fig, nv,  "#00ff9d", f"Lembah ~{pd.Timestamp(nv).strftime('%d %b')}")

            # Volume
            cv = ["#00ff9d" if df["Close_IDR"].iloc[i] >= df["Open_IDR"].iloc[i] else "#ff3d6e"
                  for i in range(len(df))]
            fig.add_trace(go.Bar(x=df["Date"], y=df["Volume"], marker_color=cv,
                                 name="Volume", opacity=0.7), row=2, col=1)

            fig.update_layout(
                template="plotly_dark", height=580,
                paper_bgcolor="rgba(0,0,0,0)", plot_bgcolor="rgba(8,12,18,1)",
                hovermode="x unified", dragmode="pan",
                legend=dict(orientation="h", y=1.01, x=0, font=dict(size=10)),
                xaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.04)",
                           rangeslider=dict(visible=False), type="date"),
                yaxis=dict(showgrid=True, gridcolor="rgba(255,255,255,0.05)",
                           tickprefix="Rp ", tickformat=",.0f", side="right"),
                xaxis2=dict(showgrid=False, type="date"),
                yaxis2=dict(showgrid=False, title=""),
                margin=dict(l=10, r=120, t=40, b=10),
            )
            st.plotly_chart(fig, use_container_width=True,
                            config=dict(scrollZoom=True, displayModeBar=True,
                                        modeBarButtonsToAdd=["drawline","eraseshape"]))

            # ── 3 KOLOM UTAMA ─────────────────────────────────────────
            col1, col2, col3 = st.columns([1, 1, 1])

            # ── COL 1: TRADING PLAN ───────────────────────────────────
            with col1:
                st.markdown('<div class="sec-header">Trading Plan</div>', unsafe_allow_html=True)
                st.markdown(f"""
                <div class="tp-card tp-buy">
                  <div><div class="tp-label tp-label-buy">AREA BELI IDEAL</div>
                  <div class="tp-val">Rp {sup:,.0f} – Rp {sup*1.02:,.0f}</div></div>
                </div>
                <div class="tp-card tp-buy">
                  <div><div class="tp-label tp-label-buy">BUY BREAKOUT</div>
                  <div class="tp-val">Tembus Rp {res*1.01:,.0f}</div></div>
                </div>
                <div class="tp-card tp-sell">
                  <div><div class="tp-label tp-label-sell">TAKE PROFIT</div>
                  <div class="tp-val">Rp {res*0.98:,.0f}</div></div>
                </div>
                <div class="tp-card tp-sell">
                  <div><div class="tp-label tp-label-sell">CUT LOSS</div>
                  <div class="tp-val">Rp {sup*0.96:,.0f}</div></div>
                </div>
                <div class="tp-card tp-wait">
                  <div><div class="tp-label tp-label-wait">RISK / REWARD</div>
                  <div class="tp-val">1 : {rr_num:.1f}</div></div>
                </div>
                """, unsafe_allow_html=True)

            # ── COL 2: FUNDAMENTAL ────────────────────────────────────
            with col2:
                st.markdown('<div class="sec-header">Fundamental</div>', unsafe_allow_html=True)

                def fval(v, fmt=".1f", suffix=""):
                    return f"{v:{fmt}}{suffix}" if v is not None else "--"
                def fcls(v, lo, hi):
                    if v is None: return "fund-mid"
                    return "fund-ok" if v<=lo else ("fund-mid" if v<=hi else "fund-bad")
                def fclsR(v, lo, hi):  # reversed (higher=better)
                    if v is None: return "fund-mid"
                    return "fund-ok" if v>=hi else ("fund-mid" if v>=lo else "fund-bad")

                per_v = fund.get("PER"); pbv_v = fund.get("PBV")
                roe_v = fund.get("ROE"); mar_v = fund.get("Margin")
                de_v  = fund.get("DE");  bet_v = fund.get("Beta")
                div_v = fund.get("Div"); eps_v = fund.get("EPS")
                h52h  = fund.get("52H"); h52l  = fund.get("52L")
                mc    = fund.get("MCap"); desk = fund.get("Desk","")

                st.markdown(f"""
                <div class="fund-grid">
                  <div class="fund-row"><span class="fund-key">PER</span>
                    <span class="fund-val {fcls(per_v,15,25)}">{fval(per_v,".1f","x") if per_v else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">PBV</span>
                    <span class="fund-val {fcls(pbv_v,1,3)}">{fval(pbv_v,".2f","x") if pbv_v else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">ROE</span>
                    <span class="fund-val {fclsR(roe_v*100 if roe_v else None,10,20)}">{f"{roe_v*100:.1f}%" if roe_v else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">Net Margin</span>
                    <span class="fund-val {fclsR(mar_v*100 if mar_v else None,5,15)}">{f"{mar_v*100:.1f}%" if mar_v else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">Debt/EQ</span>
                    <span class="fund-val {fcls(de_v,60,120)}">{fval(de_v,".0f","%") if de_v is not None else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">Beta</span>
                    <span class="fund-val fund-mid">{fval(bet_v,".2f") if bet_v else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">EPS</span>
                    <span class="fund-val fund-mid">{fval(eps_v,".2f") if eps_v else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">Div Yield</span>
                    <span class="fund-val fund-ok">{f"{div_v*100:.2f}%" if div_v else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">52W High</span>
                    <span class="fund-val fund-neu">{f"Rp {h52h:,.0f}" if h52h else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">52W Low</span>
                    <span class="fund-val fund-neu">{f"Rp {h52l:,.0f}" if h52l else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">Market Cap</span>
                    <span class="fund-val fund-mid">{fmt_mcap(mc)}</span></div>
                  <div class="fund-row"><span class="fund-key">Siklus Puncak</span>
                    <span class="fund-val fund-mid">{f"{ap} hari" if ap else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">Siklus Lembah</span>
                    <span class="fund-val fund-mid">{f"{av} hari" if av else "--"}</span></div>
                  <div class="fund-row"><span class="fund-key">ATR Harian</span>
                    <span class="fund-val fund-mid">Rp {atr_v:,.0f}</span></div>
                </div>
                """, unsafe_allow_html=True)

                # Skor ring
                sk_cls = "skor-ok" if skor>=70 else ("skor-mid" if skor>=45 else "skor-bad")
                sk_lbl = "EXCELLENT" if skor>=75 else ("GOOD" if skor>=55 else ("FAIR" if skor>=35 else "WEAK"))
                st.markdown(f"""
                <div class="skor-wrap" style="margin-top:12px;">
                  <div class="skor-ring {sk_cls}">{skor}</div>
                  <div>
                    <div style="font-family:Syne,sans-serif;font-weight:700;font-size:1rem;">{sk_lbl}</div>
                    <div style="font-size:0.72rem;color:#5a6a82;">Skor Fundamental Maestro /100</div>
                    {'<div style="font-size:0.72rem;color:#5a6a82;margin-top:6px;">'+desk[:120]+"...</div>" if desk else ""}
                  </div>
                </div>
                """, unsafe_allow_html=True)

            # ── COL 3: JADWAL 7 HARI ─────────────────────────────────
            with col3:
                st.markdown('<div class="sec-header">Jadwal 7 Hari</div>', unsafe_allow_html=True)
                today_ts = pd.Timestamp.now().normalize()
                for i in range(1, 8):
                    h_ts   = today_ts + pd.Timedelta(days=i)
                    nm     = HARI_ID.get(h_ts.strftime("%A"), h_ts.strftime("%a"))
                    tgl_s  = h_ts.strftime(f"{nm}, %d %b")
                    is_buy = nv  and h_ts == pd.Timestamp(nv).normalize()
                    is_sel = np_ and h_ts == pd.Timestamp(np_).normalize()
                    if is_buy:
                        row_cls  = "sched-row sched-buy"
                        badge_cl = "badge badge-buy"; badge_t = "BELI"
                        aksi = f"Prediksi Lembah → area Rp {sup:,.0f}"
                    elif is_sel:
                        row_cls  = "sched-row sched-sell"
                        badge_cl = "badge badge-sell"; badge_t = "JUAL"
                        aksi = f"Prediksi Puncak → area Rp {res:,.0f}"
                    else:
                        row_cls  = "sched-row sched-wait"
                        badge_cl = "badge badge-wait"; badge_t = "WAIT"
                        aksi = "Pantau pergerakan"
                    st.markdown(f"""
                    <div class="{row_cls}">
                      <span class="sched-day">{tgl_s}</span>
                      <span class="{badge_cl}">{badge_t}</span>
                      <span style="font-size:0.76rem;">{aksi}</span>
                    </div>""", unsafe_allow_html=True)

            # ── BANDARMOLOGI DETAIL ───────────────────────────────────
            st.markdown('<div class="sec-header">Bandarmologi — Jejak Institusi</div>', unsafe_allow_html=True)
            b_cls = ("bandar-card bandar-akum" if bandar=="AKUMULASI"
                     else ("bandar-card bandar-dist" if bandar=="DISTRIBUSI"
                     else "bandar-card bandar-neu"))
            b_dot = ("#00ff9d" if bandar=="AKUMULASI" else ("#ff3d6e" if bandar=="DISTRIBUSI" else "#5a6a82"))
            b_desc = {
                "AKUMULASI": "Volume spike + candle bullish — tanda institusi/bandar sedang masuk & mengumpulkan posisi.",
                "DISTRIBUSI": "Volume spike + candle bearish — tanda institusi/bandar sedang melepas & menjual posisi.",
                "NETRAL":     "Volume normal, tidak ada tanda aktivitas institusi signifikan hari ini.",
            }
            last5 = df[["Date","Close_IDR","Volume","Bandar"]].tail(5).copy()
            last5["Close_IDR"] = last5["Close_IDR"].map(lambda x: f"Rp {x:,.0f}")
            last5["Volume"]    = last5["Volume"].map(lambda x: f"{x:,.0f}")
            last5["Date"]      = last5["Date"].dt.strftime("%d %b %Y")
            last5.columns      = ["Tanggal","Harga","Volume","Status Bandar"]

            bc1, bc2 = st.columns([1, 2])
            with bc1:
                st.markdown(f"""
                <div class="{b_cls}">
                  <div class="bandar-dot" style="background:{b_dot};box-shadow:0 0 10px {b_dot};"></div>
                  <div>
                    <div style="font-family:Syne,sans-serif;font-weight:700;font-size:1.1rem;">{bandar}</div>
                    <div style="font-size:0.75rem;color:#a0aec0;margin-top:4px;">{b_desc[bandar]}</div>
                    <div style="font-size:0.72rem;color:#5a6a82;margin-top:6px;">
                      Volume hari ini: {vol_l:,.0f} ({vol_ratio:.1f}x rata-rata)
                    </div>
                  </div>
                </div>
                """, unsafe_allow_html=True)
            with bc2:
                st.dataframe(last5, use_container_width=True, hide_index=True)

            # ── PLAN TRADING 1 BULAN ──────────────────────────────────
            st.markdown('<div class="sec-header">Plan Trading 1 Bulan Ke Depan</div>', unsafe_allow_html=True)
            jadwal = buat_jadwal_30hari(np_, nv, ap, av)
            if not jadwal:
                st.info("Data siklus belum cukup untuk proyeksi 1 bulan. Coba gunakan periode data 1y atau 2y.")
            else:
                # Summary stats
                n_beli = sum(1 for j in jadwal if j["type"]=="valley")
                n_jual = sum(1 for j in jadwal if j["type"]=="peak")
                m1, m2, m3, m4 = st.columns(4)
                m1.metric("Total Event", f"{len(jadwal)} event")
                m2.metric("Sinyal Beli", f"{n_beli}x", delta="Potensi entry")
                m3.metric("Sinyal Jual", f"{n_jual}x", delta="Potensi exit")
                m4.metric("Siklus Avg", f"{ap or '--'} / {av or '--'} hari", delta="Puncak / Lembah")

                # Header tabel
                st.markdown("""
                <div class="plan-header">
                  <span>TANGGAL</span><span>HARI</span><span>EVENT & AKSI</span><span>TARGET HARGA</span>
                </div>""", unsafe_allow_html=True)

                for j in jadwal:
                    tgl_s  = j["tgl"].strftime("%d %b %Y")
                    nm_h   = HARI_ID.get(j["tgl"].strftime("%A"), j["tgl"].strftime("%a"))
                    is_v   = j["type"] == "valley"
                    ev_txt = "LEMBAH — Area Beli & Akumulasi" if is_v else "PUNCAK — Area Jual & Take Profit"
                    hg_txt = f"Rp {sup:,.0f}" if is_v else f"Rp {res:,.0f}"
                    row_cl = "plan-row plan-row-buy" if is_v else "plan-row plan-row-sell"
                    color  = "#00ff9d" if is_v else "#ff3d6e"
                    st.markdown(f"""
                    <div class="{row_cl}">
                      <span style="font-weight:600;">{tgl_s}</span>
                      <span style="color:#5a6a82;">{nm_h}</span>
                      <span style="color:{color};">{ev_txt}</span>
                      <span style="font-weight:700;">{hg_txt}</span>
                    </div>""", unsafe_allow_html=True)

                # Strategi rangkuman
                st.markdown(f"""
                <div class="panel" style="margin-top:16px;">
                  <div style="font-family:Syne,sans-serif;font-weight:700;margin-bottom:12px;font-size:0.9rem;">Strategi Global 1 Bulan</div>
                  <div class="fund-grid">
                    <div class="fund-row"><span class="fund-key">Modal per Entry</span><span class="fund-val">Max 2% dari portofolio</span></div>
                    <div class="fund-row"><span class="fund-key">Stop Loss Global</span><span class="fund-val fund-bad">Rp {sup*0.95:,.0f}</span></div>
                    <div class="fund-row"><span class="fund-key">Target Konservatif</span><span class="fund-val fund-ok">Rp {res*0.97:,.0f}</span></div>
                    <div class="fund-row"><span class="fund-key">Target Agresif</span><span class="fund-val fund-ok">Rp {res*1.03:,.0f}</span></div>
                    <div class="fund-row"><span class="fund-key">Risk/Reward Ratio</span><span class="fund-val fund-mid">1 : {rr_num:.1f}</span></div>
                    <div class="fund-row"><span class="fund-key">Review Mingguan</span><span class="fund-val">Pantau volume spike</span></div>
                  </div>
                </div>
                """, unsafe_allow_html=True)

            # ── BERITA & SENTIMEN ─────────────────────────────────────
            st.markdown('<div class="sec-header">Berita & Sentimen Pasar</div>', unsafe_allow_html=True)
            with st.spinner("Mengambil berita terkini..."):
                berita = ambil_berita(ticker)

            if not berita:
                st.info("Berita tidak tersedia saat ini.")
            else:
                n_pos = n_neg = n_neu = 0
                for b in berita:
                    s = sentimen_teks(b["judul"])
                    if s == "pos": n_pos += 1
                    elif s == "neg": n_neg += 1
                    else: n_neu += 1
                    cls = {"pos":"news-pos","neg":"news-neg","neu":"news-neu"}[s]
                    badge_t = {"pos":"POSITIF","neg":"NEGATIF","neu":"NETRAL"}[s]
                    badge_c = {"pos":"#00ff9d","neg":"#ff3d6e","neu":"#5a6a82"}[s]
                    st.markdown(f"""
                    <div class="news-item {cls}">
                      <div style="display:flex;align-items:center;gap:8px;margin-bottom:4px;">
                        <span class="news-badge" style="color:{badge_c};">{badge_t}</span>
                        <span class="news-src">{b.get("sumber","")} · {b.get("tgl","")}</span>
                      </div>
                      <div>{b["judul"]}</div>
                    </div>""", unsafe_allow_html=True)

                # Sentimen summary
                total = n_pos + n_neg + n_neu
                overall = "BULLISH" if n_pos > n_neg else ("BEARISH" if n_neg > n_pos else "NETRAL")
                ov_col  = "#00ff9d" if overall=="BULLISH" else ("#ff3d6e" if overall=="BEARISH" else "#ffd166")
                st.markdown(f"""
                <div class="panel" style="margin-top:12px;display:flex;align-items:center;gap:20px;">
                  <div style="font-family:Syne,sans-serif;font-size:1.2rem;font-weight:800;color:{ov_col};">
                    {overall}
                  </div>
                  <div style="font-size:0.8rem;color:#5a6a82;">
                    Dari {total} berita: &nbsp;
                    <span style="color:#00ff9d;">{n_pos} positif</span> &nbsp;|&nbsp;
                    <span style="color:#ff3d6e;">{n_neg} negatif</span> &nbsp;|&nbsp;
                    <span style="color:#5a6a82;">{n_neu} netral</span>
                  </div>
                </div>
                """, unsafe_allow_html=True)

            # ── RINGKASAN AKHIR ───────────────────────────────────────
            st.markdown('<div class="sec-header">Kesimpulan Maestro</div>', unsafe_allow_html=True)
            over_c = "#00ff9d" if n_pos > n_neg else ("#ff3d6e" if n_neg > n_pos else "#ffd166")
            st.markdown(f"""
            <div class="panel">
              <div style="display:grid;grid-template-columns:repeat(3,1fr);gap:16px;">
                <div>
                  <div class="metric-label">Aset</div>
                  <div style="font-family:Syne,sans-serif;font-size:1.3rem;font-weight:800;">{ticker}</div>
                  <div style="font-size:0.75rem;color:#5a6a82;">{'Saham IDX' if is_jk else ('Crypto' if '-USD' in ticker else ('Forex' if '=X' in ticker else 'Saham US'))}</div>
                </div>
                <div>
                  <div class="metric-label">Harga & Perubahan</div>
                  <div style="font-family:Syne,sans-serif;font-size:1.1rem;font-weight:700;">Rp {harga:,.0f}</div>
                  <div style="color:{'#00ff9d' if pct>=0 else '#ff3d6e'};font-size:0.8rem;">{'▲' if pct>=0 else '▼'} {abs(pct):.2f}%</div>
                </div>
                <div>
                  <div class="metric-label">Sentimen Pasar</div>
                  <div style="font-family:Syne,sans-serif;font-size:1.1rem;font-weight:700;color:{over_c};">{overall}</div>
                </div>
                <div>
                  <div class="metric-label">Puncak Berikutnya</div>
                  <div style="font-weight:600;color:#ff3d6e;">{pd.Timestamp(np_).strftime('%d %b %Y') if np_ else 'N/A'}</div>
                </div>
                <div>
                  <div class="metric-label">Lembah Berikutnya</div>
                  <div style="font-weight:600;color:#00ff9d;">{pd.Timestamp(nv).strftime('%d %b %Y') if nv else 'N/A'}</div>
                </div>
                <div>
                  <div class="metric-label">Rekomendasi</div>
                  <div style="font-weight:700;color:{'#00ff9d' if skor>=60 else ('#ffd166' if skor>=40 else '#ff3d6e')};">
                    {'LAYAK DIPERTIMBANGKAN' if skor>=60 else ('MONITOR' if skor>=40 else 'HINDARI')}
                  </div>
                </div>
              </div>
            </div>
            """, unsafe_allow_html=True)

# ══════════════════════════════════════════════════════════════════════
# MODE: SCREENER SAHAM IDX
# ══════════════════════════════════════════════════════════════════════
elif "Saham" in mode:
    if not run:
        st.markdown("<div style='color:#5a6a82;padding:40px 0;'>Masukkan budget dan klik SCAN.</div>", unsafe_allow_html=True)
    else:
        LQ45 = [
            # ── BLUECHIP / LQ45 ──────────────────────────────────────────────
            "BBCA.JK", "BBRI.JK", "BMRI.JK", "BBNI.JK", "TLKM.JK",
            "ASII.JK", "GOTO.JK", "AMMN.JK", "BREN.JK", "BRIS.JK",
            "UNVR.JK", "ICBP.JK", "PGAS.JK", "PTBA.JK", "ADRO.JK",
            "TPIA.JK", "MEDC.JK", "BRPT.JK", "KLBF.JK", "AKRA.JK",
            "AMRT.JK", "ESSA.JK", "HRUM.JK", "ITMG.JK", "MDKA.JK",
            "PGEO.JK", "SIDO.JK", "SRTG.JK", "BUKA.JK", "CPIN.JK",
            "EXCL.JK", "INDF.JK", "INKP.JK", "INTP.JK", "ISAT.JK",
            "MBMA.JK", "MTEL.JK", "MYOR.JK", "SMGR.JK", "TOWR.JK",
            "ARTO.JK", "SCMA.JK", "EMTK.JK", "ACES.JK", "INCO.JK",

            # ── PERBANKAN & KEUANGAN ─────────────────────────────────────────
            "BTPS.JK", "BJTM.JK", "BJBR.JK", "BNGA.JK", "BNLI.JK",
            "BDMN.JK", "BKSW.JK", "NISP.JK", "MEGA.JK", "AGRO.JK",
            "BBYB.JK", "BNBA.JK", "BMAS.JK", "DNAR.JK", "NAGA.JK",
            "NOBU.JK", "PNBS.JK", "RELI.JK", "SDRA.JK", "BANK.JK",

            # ── ASURANSI & MULTIFINANCE ──────────────────────────────────────
            "WOMF.JK", "BFIN.JK", "MFIN.JK", "VRNA.JK", "TRIM.JK",
            "PANS.JK", "ABMM.JK", "ADMF.JK", "CFIN.JK", "IMJS.JK",

            # ── TEKNOLOGI & DIGITAL ──────────────────────────────────────────
            "DMMX.JK", "FREN.JK", "LUCK.JK", "MIKA.JK", "MTDL.JK",
            "MLPT.JK", "ATIC.JK", "EDGE.JK", "MSTI.JK", "MSIN.JK",
            "AWAN.JK", "KOTA.JK", "AXIO.JK", "SONA.JK", "JADI.JK",

            # ── CONSUMER GOODS & RETAIL ──────────────────────────────────────
            "GGRM.JK", "HMSP.JK", "INDF.JK", "MAPI.JK", "RALS.JK",
            "HERO.JK", "MIDI.JK", "CSAP.JK", "ARGO.JK", "SSTM.JK",
            "AISA.JK", "CAMP.JK", "CEKA.JK", "DLTA.JK", "FOOD.JK",
            "GOOD.JK", "HOKI.JK", "KEJU.JK", "SKBM.JK", "ULTJ.JK",
            "BOBA.JK", "ALTO.JK", "TBLA.JK", "BUDI.JK", "MGNA.JK",

            # ── KESEHATAN & FARMASI ──────────────────────────────────────────
            "KAEF.JK", "MERK.JK", "PYFA.JK", "SIDO.JK", "TSPC.JK",
            "DVLA.JK", "INAF.JK", "HEAL.JK", "MIKA.JK", "SILO.JK",
            "PRDA.JK", "SAME.JK", "BMHS.JK", "OMED.JK", "RSGK.JK",

            # ── PROPERTI & KONSTRUKSI ────────────────────────────────────────
            "BSDE.JK", "CTRA.JK", "LPKR.JK", "PWON.JK", "SMRA.JK",
            "APLN.JK", "ARMY.JK", "ASRI.JK", "BKSL.JK", "CAKK.JK",
            "DILD.JK", "ELTY.JK", "FORZ.JK", "GMTD.JK", "GPRA.JK",
            "JRPT.JK", "KIJA.JK", "LPCK.JK", "MDLN.JK", "MKPI.JK",
            "MTLA.JK", "MMLP.JK", "NIRO.JK", "PLIN.JK", "PPRO.JK",
            "RDTX.JK", "RODA.JK", "SMDM.JK", "TARA.JK", "WIKA.JK",
            "WSKT.JK", "ADHI.JK", "PTPP.JK", "TOTL.JK", "ACST.JK",

            # ── ENERGI & PERTAMBANGAN ────────────────────────────────────────
            "ANTM.JK", "ITMG.JK", "BUMI.JK", "ENRG.JK", "DEWA.JK",
            "GEMS.JK", "KKGI.JK", "MYOH.JK", "PKPK.JK", "SMMT.JK",
            "TOBA.JK", "DOID.JK", "BOSS.JK", "MBAP.JK", "FIRE.JK",
            "GTBO.JK", "APEX.JK", "SMRU.JK", "ELSA.JK", "RUIS.JK",
            "ARTI.JK", "BIPI.JK", "LSIP.JK", "SIMP.JK", "TAPG.JK",

            # ── MINYAK SAWIT & AGRIBISNIS ────────────────────────────────────
            "AALI.JK", "BWPT.JK", "DSNG.JK", "GZCO.JK", "JAWA.JK",
            "PALM.JK", "SGRO.JK", "SMAR.JK", "SSMS.JK", "TGRA.JK",
            "MAGP.JK", "ANJT.JK", "BNRS.JK", "GOLL.JK", "INDO.JK",

            # ── INFRASTRUKTUR & UTILITAS ─────────────────────────────────────
            "JSMR.JK", "WTON.JK", "WEGE.JK", "META.JK", "TBIG.JK",
            "SUPR.JK", "KPIG.JK", "IPCC.JK", "TLKM.JK", "IPCM.JK",
            "BIRD.JK", "BLTZ.JK", "GIAA.JK", "NELY.JK", "SAFE.JK",

            # ── MANUFAKTUR & INDUSTRI ────────────────────────────────────────
            "AUTO.JK", "GJTL.JK", "INDS.JK", "NIPS.JK", "PRAS.JK",
            "SMSM.JK", "ADMG.JK", "IMAS.JK", "LPIN.JK", "MASA.JK",
            "SRIL.JK", "RICY.JK", "POLY.JK", "ERTX.JK", "ARGO.JK",
            "BATA.JK", "IKBI.JK", "KBLM.JK", "SCCO.JK", "VOKS.JK",
            "ALKA.JK", "ALMI.JK", "BTON.JK", "GDST.JK", "ISSP.JK",
            "JKSW.JK", "KRAS.JK", "LION.JK", "TBMS.JK", "ZINC.JK",

            # ── SEMEN & MATERIAL ─────────────────────────────────────────────
            "SMBR.JK", "JECC.JK", "ARNA.JK", "MARK.JK", "MLIA.JK",
            "TOTO.JK", "KIAS.JK", "AMFG.JK", "CANGK.JK","IFII.JK",

            # ── HIDDEN GEM — KAPITALISASI KECIL-MENENGAH ────────────────────
            "OILS.JK", "ANDI.JK", "CUAN.JK", "PNLF.JK", "LPGI.JK",
            "YELO.JK", "UCID.JK", "UFOE.JK", "MSKY.JK", "VICI.JK",
            "SWAT.JK", "GAMA.JK", "LABA.JK", "BHAT.JK", "BPTR.JK",
            "RCCC.JK", "MNCN.JK", "KBLV.JK", "PBSA.JK", "HATM.JK",
            "MABA.JK", "GRIA.JK", "BPII.JK", "HILL.JK", "TGUK.JK",
            "PMMP.JK", "DMAS.JK", "URBN.JK", "NASI.JK", "RAFI.JK",

            # ── MEDIA & ENTERTAINMENT ────────────────────────────────────────
            "FILM.JK", "IPTV.JK", "KREN.JK", "LINK.JK", "MDIA.JK",
            "MSIN.JK", "NFCX.JK", "CENT.JK", "ABBA.JK", "RIMO.JK",
        ]
        hasil = []
        bar   = st.progress(0, text="Scanning saham...")
        for i, tk in enumerate(LQ45):
            df, is_jk, _ = tarik_data(tk, "6mo")
            if df is not None and not df.empty:
                h   = float(df["Close_IDR"].iloc[-1])
                lot = h * 100
                if lot <= budget:
                    f   = ambil_fundamental(tk)
                    sk  = hitung_skor(f)
                    sup = float(df["Support"].dropna().iloc[-1]) if not df["Support"].dropna().empty else h*0.95
                    res = float(df["Resistance"].dropna().iloc[-1]) if not df["Resistance"].dropna().empty else h*1.05
                    pct = (h - float(df["Close_IDR"].iloc[-2]))/max(float(df["Close_IDR"].iloc[-2]),1)*100 if len(df)>1 else 0
                    hasil.append({"Rank":"","Ticker":tk,
                                  "Nama":(f.get("Nama") or tk)[:22],
                                  "Harga (Rp)":h, "1 Lot (Rp)":lot,
                                  "Chg%":round(pct,2), "Skor":sk,
                                  "PER":round(float(f["PER"]),1) if f.get("PER") else None,
                                  "PBV":round(float(f["PBV"]),2) if f.get("PBV") else None,
                                  "ROE%":round(float(f["ROE"])*100,1) if f.get("ROE") else None,
                                  "Support":sup,"Resistance":res,
                                  "Sektor":f.get("Sektor","--")})
            bar.progress((i+1)/len(LQ45), text=f"Scanning {tk}...")
        bar.empty()
        if hasil:
            df_h = pd.DataFrame(hasil).sort_values("Skor",ascending=False).reset_index(drop=True)
            df_h.index += 1; df_h.index.name = "Rank"
            st.markdown(f'<div class="sec-header">Hasil Screener — {len(df_h)} saham lolos budget Rp {budget:,.0f}</div>', unsafe_allow_html=True)
            st.dataframe(df_h.style.format({
                "Harga (Rp)":"Rp {:,.0f}","1 Lot (Rp)":"Rp {:,.0f}",
                "Support":"Rp {:,.0f}","Resistance":"Rp {:,.0f}",
                "Chg%":"{:+.2f}%","Skor":"{:.0f}",
                "PER":lambda x: f"{x:.1f}x" if x else "--",
                "PBV":lambda x: f"{x:.2f}x" if x else "--",
                "ROE%":lambda x: f"{x:.1f}%" if x else "--",
            }), use_container_width=True)
        else:
            st.warning(f"Tidak ada saham yang lolos budget Rp {budget:,.0f}. Coba naikkan budget.")

# ══════════════════════════════════════════════════════════════════════
# MODE: SCREENER CRYPTO
# ══════════════════════════════════════════════════════════════════════
elif "Crypto" in mode:
    if not run:
        st.markdown("<div style='color:#5a6a82;padding:40px 0;'>Masukkan budget dan klik SCAN.</div>", unsafe_allow_html=True)
    else:
        CRYPTO = [
                  # ── LARGE CAP ───────────────────────────────────────
                  "BTC-USD", "ETH-USD", "BNB-USD", "SOL-USD", "XRP-USD",
                  "ADA-USD", "AVAX-USD", "DOGE-USD", "TRX-USD", "TON-USD",
                  # ── MID CAP POPULER ─────────────────────────────────
                  "LINK-USD", "DOT-USD", "MATIC-USD", "LTC-USD", "ATOM-USD",
                  "UNI-USD", "NEAR-USD", "APT-USD", "OP-USD", "ARB-USD",
                  "IMX-USD", "INJ-USD", "SUI-USD", "SEI-USD", "JUP-USD",
                  # ── HIDDEN GEM POTENSIAL ─────────────────────────────
                  "FET-USD", "RENDER-USD", "WLD-USD", "ONDO-USD", "PENDLE-USD",
                  "STRK-USD", "PYTH-USD", "JTO-USD", "ETHFI-USD", "EIGEN-USD",
                  "MANTA-USD", "ALT-USD", "PORTAL-USD", "PIXEL-USD", "SAFE-USD",
                  # ── DEFI & UTILITY ──────────────────────────────────
                  "AAVE-USD", "MKR-USD", "CRV-USD", "LDO-USD", "RPL-USD",
                  "SNX-USD", "BAL-USD", "COMP-USD", "SUSHI-USD", "1INCH-USD",
                  # ── KOMODITAS CRYPTO ────────────────────────────────
                  "PAXG-USD", "GOLD-USD", "SILVER-USD", "OIL-USD", "NGAS-USD",]
        hasil = []
        bar   = st.progress(0, text="Scanning crypto...")
        for i, tk in enumerate(CRYPTO):
            df, _, kurs = tarik_data(tk, "6mo")
            if df is not None and not df.empty:
                h   = float(df["Close_IDR"].iloc[-1])
                pct = (h - float(df["Close_IDR"].iloc[-2]))/max(float(df["Close_IDR"].iloc[-2]),1)*100 if len(df)>1 else 0
                ap, np_, av, nv, _, _ = hitung_siklus(df)
                hasil.append({"Ticker":tk, "Harga (Rp)":h,
                              "Chg%":round(pct,2),
                              "Budget?":"[v] Ya" if h<=budget else "[x] Tidak",
                              "Jumlah dg Budget":f"{budget/h:.4f} unit" if h<=budget else "--",
                              "Siklus Puncak":f"{ap} hari" if ap else "--",
                              "Siklus Lembah":f"{av} hari" if av else "--",
                              "Next Peak":pd.Timestamp(np_).strftime('%d %b') if np_ else "--",
                              "Next Valley":pd.Timestamp(nv).strftime('%d %b') if nv else "--"})
            bar.progress((i+1)/len(CRYPTO), text=f"Scanning {tk}...")
        bar.empty()
        df_h = pd.DataFrame(hasil).sort_values("Harga (Rp)").reset_index(drop=True)
        df_h.index += 1
        st.markdown(f'<div class="sec-header">Hasil Screener Crypto — {len(df_h)} aset</div>', unsafe_allow_html=True)
        st.dataframe(df_h.style.format({"Harga (Rp)":"Rp {:,.2f}","Chg%":"{:+.2f}%"}),
                     use_container_width=True)

# ══════════════════════════════════════════════════════════════════════
# MODE: SCREENER FOREX
# ══════════════════════════════════════════════════════════════════════
elif "Forex" in mode:
    if not run:
        st.markdown("<div style='color:#5a6a82;padding:40px 0;'>Masukkan budget dan klik SCAN.</div>", unsafe_allow_html=True)
    else:
        FOREX = [
                # ── MAJOR PAIRS ─────────────────────────────────────
                "EURUSD=X", "GBPUSD=X", "USDJPY=X", "AUDUSD=X", "USDCHF=X",
                "USDCAD=X", "NZDUSD=X", "EURGBP=X", "EURJPY=X", "GBPJPY=X",
                # ── ASIAN & EM PAIRS ────────────────────────────────
                "USDIDR=X", "USDSGD=X", "USDMYR=X", "USDTHB=X", "USDPHP=X",
                "USDCNY=X", "USDINR=X", "USDKRW=X", "USDVND=X", "USDBRL=X",
                # ── KOMODITAS ───────────────────────────────────────
                "XAUUSD=X",  # Emas
                "XAGUSD=X",  # Perak
                "XPTUSD=X",  # Platinum
                "XPDUSD=X",  # Palladium
                # ── KOMODITAS VIA ETF/FUTURES YFINANCE ──────────────
                "CL=F",      # Crude Oil WTI
                "BZ=F",      # Brent Crude Oil
                "GC=F",      # Gold Futures
                "SI=F",      # Silver Futures
                "HG=F",      # Copper Futures
                "NG=F",      # Natural Gas
                "ZC=F",      # Corn Futures
                "ZW=F",      # Wheat Futures
                "ZS=F",      # Soybean Futures
                "KC=F",      # Coffee Futures
                "CC=F",      # Cocoa Futures
                "SB=F",]
        hasil = []
        bar   = st.progress(0, text="Scanning forex...")
        for i, tk in enumerate(FOREX):
            df, _, kurs = tarik_data(tk, "6mo")
            if df is not None and not df.empty:
                h   = float(df["Close_IDR"].iloc[-1])
                pct = (h - float(df["Close_IDR"].iloc[-2]))/max(float(df["Close_IDR"].iloc[-2]),1)*100 if len(df)>1 else 0
                hasil.append({"Instrumen":tk, "Harga (Rp)":h,
                              "Chg%":round(pct,2),
                              "Budget?":"[v] Ya" if h<=budget else "[x] Tidak"})
            bar.progress((i+1)/len(FOREX), text=f"Scanning {tk}...")
        bar.empty()
        df_h = pd.DataFrame(hasil).sort_values("Harga (Rp)").reset_index(drop=True)
        df_h.index += 1
        st.markdown(f'<div class="sec-header">Hasil Screener Forex & Komoditas</div>', unsafe_allow_html=True)
        st.dataframe(df_h.style.format({"Harga (Rp)":"Rp {:,.2f}","Chg%":"{:+.2f}%"}),
                     use_container_width=True)
'''

# ── REQUIREMENTS ────────────────────────────────────────────────────────
REQUIREMENTS = """\
streamlit>=1.30.0
yfinance>=0.2.38
pandas>=2.0.0
numpy>=1.24.0
scipy>=1.11.0
plotly>=5.18.0
requests>=2.31.0
beautifulsoup4>=4.12.0
lxml>=4.9.0
"""

CONFIG_TOML = """\
[theme]
base = "dark"
primaryColor = "#00c8ff"
backgroundColor = "#080c12"
secondaryBackgroundColor = "#0d1420"
textColor = "#e8edf5"
font = "monospace"
"""

BAT_CODE = """\
@echo off
title Maestro Trading Intelligence v3.0
color 0B
echo ============================================================
echo   MAESTRO TRADING INTELLIGENCE v3.0
echo ============================================================
pip install streamlit yfinance pandas plotly scipy numpy requests beautifulsoup4 lxml >nul 2>&1
echo Membuka aplikasi di browser...
streamlit run Maestro_App.py
pause
"""

SH_CODE = """\
#!/bin/bash
echo "Maestro Trading Intelligence v3.0"
pip install streamlit yfinance pandas plotly scipy numpy requests beautifulsoup4 lxml -q
streamlit run Maestro_App.py
"""

PANDUAN = """\
=======================================================================
 PANDUAN BUILD APK ANDROID - MAESTRO TRADING INTELLIGENCE
=======================================================================

CARA 1: WEB (Streamlit Cloud) - PALING MUDAH - 15 MENIT - GRATIS
-----------------------------------------------------------------------
1. Daftar di github.com (gratis)
2. Buat repo baru "maestro-trading" -> set PUBLIC
3. Upload: Maestro_App.py + requirements.txt
4. Buka share.streamlit.io -> Login GitHub
5. New App -> pilih repo -> Main file: Maestro_App.py -> Deploy
6. Dapat URL publik -> buka di HP = langsung jalan!

CARA 2: APK ANDROID (wrap URL ke APK) - GRATIS
-----------------------------------------------------------------------
Setelah dapat URL dari Cara 1:
1. Buka gonative.io ATAU websitetoapk.com
2. Masukkan URL Streamlit Anda
3. Download APK -> install di Android
   (Settings -> Install unknown apps -> izinkan)

CARA 3: TERMUX (offline di HP) - TANPA CLOUD
-----------------------------------------------------------------------
1. Install Termux dari F-Droid.org
2. pkg update && pkg install python
3. pip install streamlit yfinance pandas numpy scipy plotly requests bs4 lxml
4. streamlit run Maestro_App.py
5. Buka di browser HP: http://localhost:8501

TIP: Untuk homescreen seperti app native:
Chrome -> buka URL Streamlit -> menu (...) -> "Add to Home Screen"
=======================================================================
"""

# ── SIMPAN SEMUA FILE ────────────────────────────────────────────────────
with open("Maestro_App.py",   "w", encoding="utf-8") as f: f.write(APP_CODE)
with open("requirements.txt", "w", encoding="utf-8") as f: f.write(REQUIREMENTS)
with open("Mulai_Maestro.bat","w", encoding="utf-8") as f: f.write(BAT_CODE)
with open("mulai_maestro.sh", "w", encoding="utf-8") as f: f.write(SH_CODE)
with open("PANDUAN_ANDROID.txt","w",encoding="utf-8") as f: f.write(PANDUAN)
os.makedirs(".streamlit", exist_ok=True)
with open(".streamlit/config.toml","w",encoding="utf-8") as f: f.write(CONFIG_TOML)

print("=" * 65)
print("  MAESTRO APP v3.0 BERHASIL DIBUILD")
print("=" * 65)
print("  File yang dihasilkan:")
print("  [1] Maestro_App.py          <- App Streamlit Premium")
print("  [2] requirements.txt        <- Dependensi")
print("  [3] .streamlit/config.toml  <- Tema dark kustom")
print("  [4] Mulai_Maestro.bat       <- Klik 2x di Windows")
print("  [5] mulai_maestro.sh        <- Linux/Mac")
print("  [6] PANDUAN_ANDROID.txt     <- Cara build APK")
print("=" * 65)
print()
print("  CARA CEPAT JALANKAN LOKAL:")
print("  Windows : klik 2x Mulai_Maestro.bat")
print("  Terminal: streamlit run Maestro_App.py")
print()
print("  CARA UBAH PERIODE DEFAULT:")
print("  Cari baris: index=3 di selectbox periode")
print("  0=1mo | 1=2mo | 2=3mo | 3=6mo | 4=1y | 5=2y | 6=5y")
print("=" * 65)

  Membangun Maestro Pro Max...
  MAESTRO APP v3.0 BERHASIL DIBUILD
  File yang dihasilkan:
  [1] Maestro_App.py          <- App Streamlit Premium
  [2] requirements.txt        <- Dependensi
  [3] .streamlit/config.toml  <- Tema dark kustom
  [4] Mulai_Maestro.bat       <- Klik 2x di Windows
  [5] mulai_maestro.sh        <- Linux/Mac
  [6] PANDUAN_ANDROID.txt     <- Cara build APK

  CARA CEPAT JALANKAN LOKAL:
  Windows : klik 2x Mulai_Maestro.bat
  Terminal: streamlit run Maestro_App.py

  CARA UBAH PERIODE DEFAULT:
  Cari baris: index=3 di selectbox periode
  0=1mo | 1=2mo | 2=3mo | 3=6mo | 4=1y | 5=2y | 6=5y
